In [1]:
import pandas as pd
import os
import re

# %pip install openpyxl

In [2]:
trinucleotide_counts=pd.read_csv("../trinucleotide_count_full.csv", sep="\t")
trinucleotide_counts["file"]=trinucleotide_counts["Path"].str.split("/").str[-1]
trinucleotide_counts["folder"]=trinucleotide_counts["Path"].str.split("/").str[-2]
trinucleotide_counts["read"]=trinucleotide_counts["file"].str.split("_R._").str[0]
trinucleotide_counts["read_fixed"]=trinucleotide_counts["read"].str.upper()


In [3]:
# ON ERDA
# directory_path = '../sequencing_runs/'
# folders = [folder for folder in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, folder))]
# df_folder_structure = pd.DataFrame(folders, columns=['dir_name'])
# df_folder_structure["full_path"] = df_folder_structure["dir_name"].apply(lambda x: os.path.abspath(os.path.join(directory_path, x)))
# df_folder_structure.shotgun("sequencing_runs_paths_ERDA.csv")
df_folder_structure=pd.read_csv("../sequencing_runs_paths_ERDA.csv")
df_folder_structure

,Unnamed: 0,dir_name,full_path
0,0,seq230707_DWDQH,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
1,1,seq230315_PF2CP,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
2,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
3,3,seq240412_BRLUV,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
4,4,seq240424_T6U6R,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
5,5,seq231206_RISO2,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
6,6,seq230714_FWU63,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
7,7,seq230123_ISEQ2,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
8,8,seq221122_ISEQ1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...
9,9,seq221219_K93FG,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...


In [4]:
metadata_crushed_seq_controls =pd.read_excel("../EXCEL_FILES/crushed_metadata_20260227.xlsx", sheet_name="seq_controls")
metadata_shotgun_2022_amplicon =pd.read_excel("../EXCEL_FILES/shotgun_metadata_20250117.xlsx", sheet_name="2022_amplicons")
metadata_shotgun_2023_amplicon =pd.read_excel("../EXCEL_FILES/shotgun_metadata_20250117.xlsx", sheet_name="2023_amplicons")
metadata_washed_2020 =pd.read_excel("../EXCEL_FILES/washed_individually_metadata_DK_20250327.xlsx", sheet_name="2020")
metadata_shotgun_2022 =pd.read_excel("../EXCEL_FILES/shotgun_metadata_20250117.xlsx", sheet_name="2022_shotgun")
metadata_shotgun_2023 =pd.read_excel("../EXCEL_FILES/shotgun_metadata_20250117.xlsx", sheet_name="2023_shotgun")
metadata_washed_2022 =pd.read_excel("../EXCEL_FILES/washed_individually_metadata_DK_20250327.xlsx", sheet_name="2022")
metadata_crushed_2022 =pd.read_excel("../EXCEL_FILES/crushed_metadata_20260227.xlsx", sheet_name="2022")
metadata_crushed_2023_16S =pd.read_excel("../EXCEL_FILES/crushed_metadata_20260227.xlsx", sheet_name="2023_16S")
metadata_crushed_2023_ITS =pd.read_excel("../EXCEL_FILES/crushed_metadata_20260227.xlsx", sheet_name="2023_ITS")


## SEQ CONTROLS

In [5]:
print("Number of samples: "  + str(len(metadata_crushed_seq_controls)))
print("Number of sequence runs: "  + str(metadata_crushed_seq_controls["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_crushed_seq_controls["run_ID"].value_counts().to_frame()


Number of samples: 360
Number of sequence runs: 360

Number of sequence runs per folder: 


,count
run_ID,
seq230707_DWDQH,24
seq230526_JRV78,24
seq230714_FWU63,24
seq230710_GW4I8,24
seq240412_BRLUV,24
seq240209_KSVAZ,24
seq240508_LNERQ,24
seq231107_RISO1,24
seq240222_RQS8X,24


In [6]:
metadata_crushed_seq_controls_full=metadata_crushed_seq_controls.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_crushed_seq_controls_full=metadata_crushed_seq_controls_full[metadata_crushed_seq_controls_full["seq_sample_ID"].notna()]

metadata_crushed_seq_controls_full["number_bio"]=metadata_crushed_seq_controls_full["bio_sample_ID"].str.split("-c-").str[1]
metadata_crushed_seq_controls_full

metadata_crushed_seq_controls_full["number_sample"]=metadata_crushed_seq_controls_full["correct_seq_sample_ID"].str.split("-c-").str[1].str.rsplit( "-",n=1).str[0]
metadata_crushed_seq_controls_full

metadata_crushed_seq_controls_full['suggested_correct_seq_sample_ID'] = \
    metadata_crushed_seq_controls_full['correct_seq_sample_ID'].fillna(metadata_crushed_seq_controls_full['seq_sample_ID'])
metadata_crushed_seq_controls_full['suggested_correct_seq_sample_ID'] = metadata_crushed_seq_controls_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_crushed_seq_controls_full=metadata_crushed_seq_controls_full.sort_values("run_ID")

metadata_crushed_seq_controls_full['suggested_correct_seq_sample_ID'] = metadata_crushed_seq_controls_full.apply(
    lambda row: row['suggested_correct_seq_sample_ID'].replace(str(row["number_sample"]), str(row["number_bio"]))
    if pd.notna(row['suggested_correct_seq_sample_ID']) else row['suggested_correct_seq_sample_ID'],
    axis=1)

metadata_crushed_seq_controls_full



,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,seq_sample_ID,run_ID,note,correct_seq_sample_ID,Unnamed: 0,dir_name,full_path,number_bio,number_sample,suggested_correct_seq_sample_ID
35,A-230622-c-66,NaN,66,2022-06-23,deep,3,18,NaN,B7,y,...,A-230622-ITS-c-66,seq230526_JRV78,Alex_May_have_thawed_before_freezedrying,A-230622-ITS-c-66-JRV78,15,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,66,66,A-230622-ITS-c-66-JRV78
26,A-230622-c-61-3,NaN,61,2022-06-23,deep,3,13,NaN,A7,n,...,A-230622-ITS-c-61-3,seq230526_JRV78,NaN,A-230622-ITS-c-61-3-JRV78,15,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,61-3,61-3,A-230622-ITS-c-61-3-JRV78
27,A-230622-c-62-1,NaN,62,2022-06-23,deep,3,14,NaN,A16,n,...,A-230622-ITS-c-62-1,seq230526_JRV78,NaN,A-230622-ITS-c-62-1-JRV78,15,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,62-1,62-1,A-230622-ITS-c-62-1-JRV78
28,A-230622-c-62-2,NaN,62,2022-06-23,deep,3,14,NaN,A16,n,...,A-230622-ITS-c-62-2,seq230526_JRV78,NaN,A-230622-ITS-c-62-2-JRV78,15,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,62-2,62-2,A-230622-ITS-c-62-2-JRV78
29,A-230622-c-62-3,NaN,62,2022-06-23,deep,3,14,NaN,A16,n,...,A-230622-ITS-c-62-3,seq230526_JRV78,NaN,A-230622-ITS-c-62-3-JRV78,15,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,62-3,62-3,A-230622-ITS-c-62-3-JRV78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,A-230622-c-51-5,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,A-230622-16S-c-51-5,seq240826_X6DU5,NaN,A-230622-16S-c-51-5-X6DU5,13,seq240826_X6DU5,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,51-5,51-5,A-230622-16S-c-51-5-X6DU5
337,A-230622-c-51-4,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,A-230622-16S-c-51-4,seq240826_X6DU5,NaN,A-230622-16S-c-51-4-X6DU5,13,seq240826_X6DU5,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,51-4,51-4,A-230622-16S-c-51-4-X6DU5
336,A-230622-c-51-3,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,A-230622-16S-c-51-3,seq240826_X6DU5,NaN,A-230622-16S-c-51-3-X6DU5,13,seq240826_X6DU5,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,51-3,51-3,A-230622-16S-c-51-3-X6DU5
346,A-230622-c-57,NaN,57,2022-06-23,deep,2,9,NaN,B5,y,...,A-230622-16S-c-57,seq240826_X6DU5,Alex_May_have_thawed_before_freezedrying,A-230622-16S-c-57-X6DU5,13,seq240826_X6DU5,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,57,57,A-230622-16S-c-57-X6DU5


In [7]:
def list_files_with_prefix(folder_path, prefix):
    """
    Returns sorted file paths from the already-loaded
    `trinucleotide_counts` DataFrame where:
      - 'Path' contains folder_path
      - 'Path' contains prefix
    """
    try:
        if "Path" not in trinucleotide_counts.columns:
            return []
        
        matching = trinucleotide_counts["Path"].str.contains(folder_path, na=False) & \
                   trinucleotide_counts["Path"].str.contains(prefix, na=False)
        
        matching_files = (
            trinucleotide_counts.loc[matching, "Path"]
            .sort_values()
            .tolist()
        )
        
        return matching_files
    
    except Exception:
        return []

# Iterate over each row and print files
metadata_crushed_seq_controls_full["forward_fq"]=""
metadata_crushed_seq_controls_full["reverse_fq"]=""
metadata_crushed_seq_controls_full["current_seq_sample_ID"]=""
metadata_crushed_seq_controls_full["corrected_forward_fq"]=""
metadata_crushed_seq_controls_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_crushed_seq_controls_full)  # Get the total number of rows

for index, row in metadata_crushed_seq_controls_full.iterrows():
    try:
        folder_path = row['full_path']  
        prefix = row['seq_sample_ID'] + "_"  
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        if len(matching_files) >2:
            read_counts=trinucleotide_counts[trinucleotide_counts["Path"].isin(matching_files)].sort_values(by="Path")
            read_counts=read_counts[read_counts["Reads"]>0]
            if len(read_counts)==2:
                print(matching_files[0].split("/")[-2:], matching_files[2].split("/")[-1:])
                print()
                matching_files=read_counts["Path"].to_list()
        
        if len(matching_files) == 2:
            metadata_crushed_seq_controls_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_crushed_seq_controls_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_crushed_seq_controls_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_crushed_seq_controls_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_crushed_seq_controls_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1
    
print("\nProcessing complete!")


['seq240424_T6U6R', 'A-230622-16S-c-58-2_S17_L001_R1_001.fastq.gz'] ['A-230622-16S-c-58-2_S187_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-51-5_S177_L001_R1_001.fastq.gz'] ['A-230622-16S-c-51-5_S7_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-6_S198_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-6_S28_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-5_S197_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-5_S27_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-4_S196_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-4_S26_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-3_S195_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-3_S25_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-2_S194_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-2_S24_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-59-1_S193_L001_R1_001.fastq.gz'] ['A-230622-16S-c-59-1_S23_L001_R1_001.fastq.gz']

['seq240424_T6U6R', 'A-230622-16S-c-58-7_S192_L00

In [8]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_crushed_seq_controls_full = metadata_crushed_seq_controls_full[
    [col for col in metadata_crushed_seq_controls_full.columns if col not in columns_to_move] + columns_to_move
]
# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_crushed_seq_controls_full['forward_fq'] = metadata_crushed_seq_controls_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_crushed_seq_controls_full['reverse_fq'] = metadata_crushed_seq_controls_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_crushed_seq_controls_full["SUGGESTED_EQUALS_CORRECT"] = metadata_crushed_seq_controls_full["suggested_correct_seq_sample_ID"] == metadata_crushed_seq_controls_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_crushed_seq_controls_full["ALREADY_CORRECT"] = metadata_crushed_seq_controls_full["suggested_correct_seq_sample_ID"] == metadata_crushed_seq_controls_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_crushed_seq_controls_full = metadata_crushed_seq_controls_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_crushed_seq_controls_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,number_sample,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
178,A-230622-c-62-5,NaN,62,2022-06-23,deep,3,14,NaN,A16,n,...,63-5,washed-1-control-A-230622-ITS-c-63-5_S258_L001...,washed-1-control-A-230622-ITS-c-63-5_S258_L001...,A-230622-ITS-c-62-5-RISO1_S258_L001_R1_001.fas...,A-230622-ITS-c-62-5-RISO1_S258_L001_R2_001.fas...,washed-1-control-A-230622-ITS-c-63-5,A-230622-ITS-c-63-5-RISO1,A-230622-ITS-c-62-5-RISO1,False,False
170,A-230622-c-60-3,NaN,60,2022-06-23,deep,2,12,NaN,A7,n,...,61-3,washed-1-control-A-230622-ITS-c-61-3_S250_L001...,washed-1-control-A-230622-ITS-c-61-3_S250_L001...,A-230622-ITS-c-60-3-RISO1_S250_L001_R1_001.fas...,A-230622-ITS-c-60-3-RISO1_S250_L001_R2_001.fas...,washed-1-control-A-230622-ITS-c-61-3,A-230622-ITS-c-61-3-RISO1,A-230622-ITS-c-60-3-RISO1,False,False
171,A-230622-c-61-1,NaN,61,2022-06-23,deep,3,13,NaN,A7,n,...,62-1,washed-1-control-A-230622-ITS-c-62-1_S251_L001...,washed-1-control-A-230622-ITS-c-62-1_S251_L001...,A-230622-ITS-c-61-1-RISO1_S251_L001_R1_001.fas...,A-230622-ITS-c-61-1-RISO1_S251_L001_R2_001.fas...,washed-1-control-A-230622-ITS-c-62-1,A-230622-ITS-c-62-1-RISO1,A-230622-ITS-c-61-1-RISO1,False,False
172,A-230622-c-61-2,NaN,61,2022-06-23,deep,3,13,NaN,A7,n,...,62-2,washed-1-control-A-230622-ITS-c-62-2_S252_L001...,washed-1-control-A-230622-ITS-c-62-2_S252_L001...,A-230622-ITS-c-61-2-RISO1_S252_L001_R1_001.fas...,A-230622-ITS-c-61-2-RISO1_S252_L001_R2_001.fas...,washed-1-control-A-230622-ITS-c-62-2,A-230622-ITS-c-62-2-RISO1,A-230622-ITS-c-61-2-RISO1,False,False
173,A-230622-c-61-3,NaN,61,2022-06-23,deep,3,13,NaN,A7,n,...,62-3,washed-1-control-A-230622-ITS-c-62-3_S253_L001...,washed-1-control-A-230622-ITS-c-62-3_S253_L001...,A-230622-ITS-c-61-3-RISO1_S253_L001_R1_001.fas...,A-230622-ITS-c-61-3-RISO1_S253_L001_R2_001.fas...,washed-1-control-A-230622-ITS-c-62-3,A-230622-ITS-c-62-3-RISO1,A-230622-ITS-c-61-3-RISO1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,A-230622-c-51-5,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,51-5,A-230622-16S-c-51-5_S145_L001_R1_001.fastq.gz,A-230622-16S-c-51-5_S145_L001_R2_001.fastq.gz,A-230622-16S-c-51-5-X6DU5_S145_L001_R1_001.fas...,A-230622-16S-c-51-5-X6DU5_S145_L001_R2_001.fas...,A-230622-16S-c-51-5,A-230622-16S-c-51-5-X6DU5,A-230622-16S-c-51-5-X6DU5,True,False
337,A-230622-c-51-4,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,51-4,A-230622-16S-c-51-4_S144_L001_R1_001.fastq.gz,A-230622-16S-c-51-4_S144_L001_R2_001.fastq.gz,A-230622-16S-c-51-4-X6DU5_S144_L001_R1_001.fas...,A-230622-16S-c-51-4-X6DU5_S144_L001_R2_001.fas...,A-230622-16S-c-51-4,A-230622-16S-c-51-4-X6DU5,A-230622-16S-c-51-4-X6DU5,True,False
336,A-230622-c-51-3,NaN,51,2022-06-23,deep,1,3,NaN,A5,n,...,51-3,A-230622-16S-c-51-3_S143_L001_R1_001.fastq.gz,A-230622-16S-c-51-3_S143_L001_R2_001.fastq.gz,A-230622-16S-c-51-3-X6DU5_S143_L001_R1_001.fas...,A-230622-16S-c-51-3-X6DU5_S143_L001_R2_001.fas...,A-230622-16S-c-51-3,A-230622-16S-c-51-3-X6DU5,A-230622-16S-c-51-3-X6DU5,True,False
346,A-230622-c-57,NaN,57,2022-06-23,deep,2,9,NaN,B5,y,...,57,A-230622-16S-c-57_S153_L001_R1_001.fastq.gz,A-230622-16S-c-57_S153_L001_R2_001.fastq.gz,A-230622-16S-c-57-X6DU5_S153_L001_R1_001.fastq.gz,A-230622-16S-c-57-X6DU5_S153_L001_R2_001.fastq.gz,A-230622-16S-c-57,A-230622-16S-c-57-X6DU5,A-230622-16S-c-57-X6DU5,True,False


In [9]:
filtered_metadata_crushed_seq_controls_full=metadata_crushed_seq_controls_full[
    metadata_crushed_seq_controls_full.apply(
        lambda row: row['amplicon'] not in row['suggested_correct_seq_sample_ID'], axis=1
    )
]
filtered_metadata_crushed_seq_controls_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,number_sample,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT


## SHOTGUN 2022 AMPLICON

In [10]:
print("Number of samples: "  + str(len(metadata_shotgun_2022_amplicon)))
print("Number of sequence runs: "  + str(metadata_shotgun_2022_amplicon["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_shotgun_2022_amplicon["run_ID"].value_counts()


Number of samples: 148
Number of sequence runs: 144

Number of sequence runs per folder: 


run_ID
seq230714_FWU63    72
seq230710_GW4I8    72
Name: count, dtype: int64

In [11]:
metadata_shotgun_2022_amplicon_full=metadata_shotgun_2022_amplicon.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_shotgun_2022_amplicon_full=metadata_shotgun_2022_amplicon_full[metadata_shotgun_2022_amplicon_full["seq_sample_ID"].notna()]
metadata_shotgun_2022_amplicon_full['suggested_correct_seq_sample_ID'] = \
    metadata_shotgun_2022_amplicon_full['correct_seq_sample_ID'].fillna(metadata_shotgun_2022_amplicon_full['seq_sample_ID'])
metadata_shotgun_2022_amplicon_full['suggested_correct_seq_sample_ID'] = metadata_shotgun_2022_amplicon_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_shotgun_2022_amplicon_full=metadata_shotgun_2022_amplicon_full.sort_values("run_ID")


In [12]:


# Iterate over each row and print files
metadata_shotgun_2022_amplicon_full["forward_fq"]=""
metadata_shotgun_2022_amplicon_full["reverse_fq"]=""
metadata_shotgun_2022_amplicon_full["current_seq_sample_ID"]=""
metadata_shotgun_2022_amplicon_full["corrected_forward_fq"]=""
metadata_shotgun_2022_amplicon_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_shotgun_2022_amplicon_full)  # Get the total number of rows

for index, row in metadata_shotgun_2022_amplicon_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        if len(matching_files) == 2:
            metadata_shotgun_2022_amplicon_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_shotgun_2022_amplicon_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_shotgun_2022_amplicon_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_shotgun_2022_amplicon_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_shotgun_2022_amplicon_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2


        if len(matching_files) >2:    
            grouped_files = defaultdict(list)
            
            # Loop over each file and group by the part before R1/R2
            for file in matching_files:
                # Split the file name at R1 or R2
                base_name = file.split("/")[-1].rsplit("_R", 1)[0]  # Split from the last occurrence of "_R"
                grouped_files[base_name].append(file.split("/")[-1])
            
            metadata_shotgun_2022_amplicon_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", grouped_files[base_name][0].split("/")[-1])[0]
     
            
            # Print the grouped files
            m=0
            for base_name, files in grouped_files.items():      
                if len(files)==2:
            
                    if m==0:
                        # print(grouped_files[base_name][0])
                        metadata_shotgun_2022_amplicon_full.loc[index, "forward_fq"] = [grouped_files[base_name][0]]
                        metadata_shotgun_2022_amplicon_full.loc[index, "reverse_fq"] = [grouped_files[base_name][1]]
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2022_amplicon_full.loc[index, "corrected_forward_fq"] = [correct_seq_sample_ID + ending_R1]
                        metadata_shotgun_2022_amplicon_full.loc[index, "corrected_reverse_fq"] = [correct_seq_sample_ID + ending_R2]
            
            
                    else:
                        metadata_shotgun_2022_amplicon_full.loc[index, "forward_fq"].append(grouped_files[base_name][0])
                        metadata_shotgun_2022_amplicon_full.loc[index, "reverse_fq"].append(grouped_files[base_name][1])
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2022_amplicon_full.loc[index, "corrected_forward_fq"].append(correct_seq_sample_ID + ending_R1)
                        metadata_shotgun_2022_amplicon_full.loc[index, "corrected_reverse_fq"].append(correct_seq_sample_ID + ending_R2)
            
                
                    m=m+1
                else:
                    print("ERROR", base_name, files)

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1
    
# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Processing... 100.00% complete
Processing complete!


In [13]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_shotgun_2022_amplicon_full = metadata_shotgun_2022_amplicon_full[
    [col for col in metadata_shotgun_2022_amplicon_full.columns if col not in columns_to_move] + columns_to_move
]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_shotgun_2022_amplicon_full["SUGGESTED_EQUALS_CORRECT"] = metadata_shotgun_2022_amplicon_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2022_amplicon_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_shotgun_2022_amplicon_full["ALREADY_CORRECT"] = metadata_shotgun_2022_amplicon_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2022_amplicon_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_shotgun_2022_amplicon_full = metadata_shotgun_2022_amplicon_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_shotgun_2022_amplicon_full

,bio_sample_ID,sample_date,sample_type,block,Plot,subplot,ID_on_field,spray,fertilizer,Cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
75,R-040722-s-2G,2022-07-04,timeline,NaN,2.0,NaN,A7,yes,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-2G_S77_L001_R1_001.fastq.gz,R-040722-ITS-s-2G_S77_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-2G,NaN,R-040722-ITS-s-2G,False,False
76,R-040722-s-3G,2022-07-04,timeline,NaN,3.0,NaN,B5,no,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-3G_S80_L001_R1_001.fastq.gz,R-040722-ITS-s-3G_S80_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-3G,NaN,R-040722-ITS-s-3G,False,False
77,R-040722-s-4G,2022-07-04,timeline,NaN,4.0,NaN,B7,no,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-4G_S83_L001_R1_001.fastq.gz,R-040722-ITS-s-4G_S83_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-4G,NaN,R-040722-ITS-s-4G,False,False
78,R-040722-s-5G,2022-07-04,timeline,NaN,5.0,NaN,B16,no,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-5G_S86_L001_R1_001.fastq.gz,R-040722-ITS-s-5G_S86_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-5G,NaN,R-040722-ITS-s-5G,False,False
79,R-040722-s-6G,2022-07-04,timeline,NaN,6.0,NaN,A5,yes,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-6G_S89_L001_R1_001.fastq.gz,R-040722-ITS-s-6G_S89_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-6G,NaN,R-040722-ITS-s-6G,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45,R-070722-s-22H,2022-07-07,timeline,NaN,22.0,NaN,B5,no,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-22H_S141_L001_R1_001.fastq.gz,R-070722-16S-s-22H_S141_L001_R2_001.fastq.gz,16S-R-07072-s-22H,R-070722-16S-s-22H,R-070722-16S-s-22H,True,False
46,R-070722-s-23H,2022-07-07,timeline,NaN,23.0,NaN,B16,no,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-23H_S143_L001_R1_001.fastq.gz,R-070722-16S-s-23H_S143_L001_R2_001.fastq.gz,16S-R-07072-s-23H,R-070722-16S-s-23H,R-070722-16S-s-23H,True,False
47,R-070722-s-24H,2022-07-07,timeline,NaN,24.0,NaN,A5,yes,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-24H_S146_L001_R1_001.fastq.gz,R-070722-16S-s-24H_S146_L001_R2_001.fastq.gz,16S-R-07072-s-24H,R-070722-16S-s-24H,R-070722-16S-s-24H,True,False
44,R-070722-s-21H,2022-07-07,timeline,NaN,21.0,NaN,B7,no,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-21H_S138_L001_R1_001.fastq.gz,R-070722-16S-s-21H_S138_L001_R2_001.fastq.gz,16S-R-07072-s-21H,R-070722-16S-s-21H,R-070722-16S-s-21H,True,False


In [14]:
filtered_metadata_shotgun_2022_amplicon_full=metadata_shotgun_2022_amplicon_full[
    metadata_shotgun_2022_amplicon_full.apply(
        lambda row: row['amplicon'] not in row['suggested_correct_seq_sample_ID'], axis=1
    )
]
metadata_shotgun_2022_amplicon_full

,bio_sample_ID,sample_date,sample_type,block,Plot,subplot,ID_on_field,spray,fertilizer,Cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
75,R-040722-s-2G,2022-07-04,timeline,NaN,2.0,NaN,A7,yes,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-2G_S77_L001_R1_001.fastq.gz,R-040722-ITS-s-2G_S77_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-2G,NaN,R-040722-ITS-s-2G,False,False
76,R-040722-s-3G,2022-07-04,timeline,NaN,3.0,NaN,B5,no,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-3G_S80_L001_R1_001.fastq.gz,R-040722-ITS-s-3G_S80_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-3G,NaN,R-040722-ITS-s-3G,False,False
77,R-040722-s-4G,2022-07-04,timeline,NaN,4.0,NaN,B7,no,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-4G_S83_L001_R1_001.fastq.gz,R-040722-ITS-s-4G_S83_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-4G,NaN,R-040722-ITS-s-4G,False,False
78,R-040722-s-5G,2022-07-04,timeline,NaN,5.0,NaN,B16,no,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-5G_S86_L001_R1_001.fastq.gz,R-040722-ITS-s-5G_S86_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-5G,NaN,R-040722-ITS-s-5G,False,False
79,R-040722-s-6G,2022-07-04,timeline,NaN,6.0,NaN,A5,yes,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-040722-ITS-s-6G_S89_L001_R1_001.fastq.gz,R-040722-ITS-s-6G_S89_L001_R2_001.fastq.gz,ITS-R-040722-ITS-s-6G,NaN,R-040722-ITS-s-6G,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45,R-070722-s-22H,2022-07-07,timeline,NaN,22.0,NaN,B5,no,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-22H_S141_L001_R1_001.fastq.gz,R-070722-16S-s-22H_S141_L001_R2_001.fastq.gz,16S-R-07072-s-22H,R-070722-16S-s-22H,R-070722-16S-s-22H,True,False
46,R-070722-s-23H,2022-07-07,timeline,NaN,23.0,NaN,B16,no,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-23H_S143_L001_R1_001.fastq.gz,R-070722-16S-s-23H_S143_L001_R2_001.fastq.gz,16S-R-07072-s-23H,R-070722-16S-s-23H,R-070722-16S-s-23H,True,False
47,R-070722-s-24H,2022-07-07,timeline,NaN,24.0,NaN,A5,yes,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-24H_S146_L001_R1_001.fastq.gz,R-070722-16S-s-24H_S146_L001_R2_001.fastq.gz,16S-R-07072-s-24H,R-070722-16S-s-24H,R-070722-16S-s-24H,True,False
44,R-070722-s-21H,2022-07-07,timeline,NaN,21.0,NaN,B7,no,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-070722-16S-s-21H_S138_L001_R1_001.fastq.gz,R-070722-16S-s-21H_S138_L001_R2_001.fastq.gz,16S-R-07072-s-21H,R-070722-16S-s-21H,R-070722-16S-s-21H,True,False


## SHOTGUN 2022

In [15]:

print("Number of samples: "  + str(len(metadata_shotgun_2022)))
print("Number of sequence runs: "  + str(metadata_shotgun_2022["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_shotgun_2022["run_ID"].value_counts()


Number of samples: 316
Number of sequence runs: 315

Number of sequence runs per folder: 


run_ID
seq230315_PF2CP    226
seq230526_LQJXR     89
Name: count, dtype: int64

In [16]:
metadata_shotgun_2022_full=metadata_shotgun_2022.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_shotgun_2022_full=metadata_shotgun_2022_full[metadata_shotgun_2022_full["seq_sample_ID"].notna()]
metadata_shotgun_2022_full['suggested_correct_seq_sample_ID'] = \
    metadata_shotgun_2022_full['correct_seq_sample_ID'].fillna(metadata_shotgun_2022_full['seq_sample_ID'])
metadata_shotgun_2022_full['suggested_correct_seq_sample_ID'] = metadata_shotgun_2022_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_shotgun_2022_full=metadata_shotgun_2022_full.sort_values("run_ID")





In [17]:
# Iterate over each row and print files
metadata_shotgun_2022_full["forward_fq"]=""
metadata_shotgun_2022_full["reverse_fq"]=""
metadata_shotgun_2022_full["current_seq_sample_ID"]=""
metadata_shotgun_2022_full["corrected_forward_fq"]=""
metadata_shotgun_2022_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_shotgun_2022_full)  # Get the total number of rows

for index, row in metadata_shotgun_2022_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        if len(matching_files) == 2:
            metadata_shotgun_2022_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_shotgun_2022_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_shotgun_2022_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_shotgun_2022_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_shotgun_2022_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2


        if len(matching_files) >2:

            
            
            grouped_files = defaultdict(list)
            
            # Loop over each file and group by the part before R1/R2
            for file in matching_files:
                # Split the file name at R1 or R2
                base_name = file.split("/")[-1].rsplit("_R", 1)[0]  # Split from the last occurrence of "_R"
                grouped_files[base_name].append(file.split("/")[-1])
            
            metadata_shotgun_2022_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", grouped_files[base_name][0].split("/")[-1])[0]
     
            
            # Print the grouped files
            m=0
            for base_name, files in grouped_files.items():      
                if len(files)==2:
            
                    if m==0:
                        # print(grouped_files[base_name][0])
                        metadata_shotgun_2022_full.loc[index, "forward_fq"] = [grouped_files[base_name][0]]
                        metadata_shotgun_2022_full.loc[index, "reverse_fq"] = [grouped_files[base_name][1]]
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2022_full.loc[index, "corrected_forward_fq"] = [correct_seq_sample_ID + ending_R1]
                        metadata_shotgun_2022_full.loc[index, "corrected_reverse_fq"] = [correct_seq_sample_ID + ending_R2]
            
            
                    else:
                        metadata_shotgun_2022_full.loc[index, "forward_fq"].append(grouped_files[base_name][0])
                        metadata_shotgun_2022_full.loc[index, "reverse_fq"].append(grouped_files[base_name][1])
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2022_full.loc[index, "corrected_forward_fq"].append(correct_seq_sample_ID + ending_R1)
                        metadata_shotgun_2022_full.loc[index, "corrected_reverse_fq"].append(correct_seq_sample_ID + ending_R2)
            
                
                    m=m+1
                else:
                    print("ERROR", base_name, files)


    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1
    
# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Error occurred for row 0: bio_sample_ID                                                          R_070622_s_1A
sample_date                                                      2022-06-07 00:00:00
sample_type                                                                 timeline
block                                                                            NaN
plot                                                                             1.0
subplot                                                                          NaN
ID_on_field                                                                      A16
spray                                                                              y
fertiliser                                                                       NaN
cultivar                                                                   rembrandt
density                                                                          NaN
harvest_date                           

In [18]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_shotgun_2022_full = metadata_shotgun_2022_full[
    [col for col in metadata_shotgun_2022_full.columns if col not in columns_to_move] + columns_to_move]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_shotgun_2022_full["SUGGESTED_EQUALS_CORRECT"] = metadata_shotgun_2022_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2022_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_shotgun_2022_full["ALREADY_CORRECT"] = metadata_shotgun_2022_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2022_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_shotgun_2022_full = metadata_shotgun_2022_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_shotgun_2022_full

,bio_sample_ID,sample_date,sample_type,block,plot,subplot,ID_on_field,spray,fertiliser,cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
0,R_070622_s_1A,2022-06-07,timeline,NaN,1.0,NaN,A16,y,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,R-070622-s-1A,False,False
144,R-040722-s-1G,2022-07-04,timeline,NaN,1.0,NaN,A16,y,NaN,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,R-040722-s-1G,False,False
145,R-040722-s-2G,2022-07-04,timeline,NaN,2.0,NaN,A7,y,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,R-040722-s-2G,False,False
146,R-040722-s-3G,2022-07-04,timeline,NaN,3.0,NaN,B5,n,NaN,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,R-040722-s-3G,False,False
147,R-040722-s-4G,2022-07-04,timeline,NaN,4.0,NaN,B7,n,NaN,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,R-040722-s-4G,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,R-000022-s-nex-lib-neg-5,R-000022-s-nex-lib-neg-5,True,False
223,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,R-000022-s-nex-lib-neg-6,R-000022-s-nex-lib-neg-6,True,False
224,DES,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,R-000022-s-hug-kit-des-neg-1,R-000022-s-hug-kit-des-neg-1,True,False
225,1xPBS,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,R-000022-s-hug-kit-pbs-neg-2,R-000022-s-hug-kit-pbs-neg-2,True,False


## Shotgun 2023 amplicon

In [19]:
print("Number of samples: "  + str(len(metadata_shotgun_2023_amplicon)))
print("Number of sequence runs: "  + str(metadata_shotgun_2023_amplicon["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_shotgun_2023_amplicon["run_ID"].value_counts()



Number of samples: 240
Number of sequence runs: 240

Number of sequence runs per folder: 


run_ID
seq240508_LNERQ    97
seq240719_NR8NI    94
seq240710_DR8PG    41
seq231107_RISO1     8
Name: count, dtype: int64

In [20]:
metadata_shotgun_2023_amplicon_full=metadata_shotgun_2023_amplicon.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_shotgun_2023_amplicon_full=metadata_shotgun_2023_amplicon_full[metadata_shotgun_2023_amplicon_full["seq_sample_ID"].notna()]
metadata_shotgun_2023_amplicon_full['suggested_correct_seq_sample_ID'] = \
    metadata_shotgun_2023_amplicon_full['correct_seq_sample_ID'].fillna(metadata_shotgun_2023_amplicon_full['seq_sample_ID'])
metadata_shotgun_2023_amplicon_full['suggested_correct_seq_sample_ID'] = metadata_shotgun_2023_amplicon_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_shotgun_2023_amplicon_full=metadata_shotgun_2023_amplicon_full.sort_values("run_ID")


In [21]:


# Iterate over each row and print files
metadata_shotgun_2023_amplicon_full["forward_fq"]=""
metadata_shotgun_2023_amplicon_full["reverse_fq"]=""
metadata_shotgun_2023_amplicon_full["current_seq_sample_ID"]=""
metadata_shotgun_2023_amplicon_full["corrected_forward_fq"]=""
metadata_shotgun_2023_amplicon_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_shotgun_2023_amplicon_full)  # Get the total number of rows

for index, row in metadata_shotgun_2023_amplicon_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        if len(matching_files) == 2:
            metadata_shotgun_2023_amplicon_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_shotgun_2023_amplicon_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_shotgun_2023_amplicon_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_shotgun_2023_amplicon_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_shotgun_2023_amplicon_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2


        if len(matching_files) >2:

            
            
            grouped_files = defaultdict(list)
            
            # Loop over each file and group by the part before R1/R2
            for file in matching_files:
                # Split the file name at R1 or R2
                base_name = file.split("/")[-1].rsplit("_R", 1)[0]  # Split from the last occurrence of "_R"
                grouped_files[base_name].append(file.split("/")[-1])
            
            metadata_shotgun_2023_amplicon_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", grouped_files[base_name][0].split("/")[-1])[0]
     
            
            # Print the grouped files
            m=0
            for base_name, files in grouped_files.items():      
                if len(files)==2:
            
                    if m==0:
                        # print(grouped_files[base_name][0])
                        metadata_shotgun_2023_amplicon_full.loc[index, "forward_fq"] = [grouped_files[base_name][0]]
                        metadata_shotgun_2023_amplicon_full.loc[index, "reverse_fq"] = [grouped_files[base_name][1]]
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2023_amplicon_full.loc[index, "corrected_forward_fq"] = [correct_seq_sample_ID + ending_R1]
                        metadata_shotgun_2023_amplicon_full.loc[index, "corrected_reverse_fq"] = [correct_seq_sample_ID + ending_R2]
            
            
                    else:
                        metadata_shotgun_2023_amplicon_full.loc[index, "forward_fq"].append(grouped_files[base_name][0])
                        metadata_shotgun_2023_amplicon_full.loc[index, "reverse_fq"].append(grouped_files[base_name][1])
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2023_amplicon_full.loc[index, "corrected_forward_fq"].append(correct_seq_sample_ID + ending_R1)
                        metadata_shotgun_2023_amplicon_full.loc[index, "corrected_reverse_fq"].append(correct_seq_sample_ID + ending_R2)
            
                
                    m=m+1
                else:
                    print("ERROR", base_name, files)


    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1
    
# After the loop, print 100% to indicate completion
print("\nProcessing complete!")


Processing... 100.00% complete
Processing complete!


In [22]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_shotgun_2023_amplicon_full = metadata_shotgun_2023_amplicon_full[
    [col for col in metadata_shotgun_2023_amplicon_full.columns if col not in columns_to_move] + columns_to_move
]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_shotgun_2023_amplicon_full["SUGGESTED_EQUALS_CORRECT"] = metadata_shotgun_2023_amplicon_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2023_amplicon_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_shotgun_2023_amplicon_full["ALREADY_CORRECT"] = metadata_shotgun_2023_amplicon_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2023_amplicon_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_shotgun_2023_amplicon_full = metadata_shotgun_2023_amplicon_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_shotgun_2023_amplicon_full#.shotgun("metadata_shotgun_19_03_2025_2023_shotgun_amplicon.csv")

,bio_sample_ID,sample_date,sample_type,block,plot,subplot,ID_on_field,spray,fertilizer,cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
239,T-210623-aphid-107D,2023-06-21,timeline,3.0,107,9.0,NaN,NaN,Normal,sheriff,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-200723-ITS-w-aphid-107D_S279_L001_R1_001.fas...,T-200723-ITS-w-aphid-107D_S279_L001_R2_001.fas...,washed-1-T-200723-ITS-w-aphid-107D,NaN,T-200723-ITS-w-aphid-107D,False,False
237,T-210623-aphid-100D,2023-06-21,timeline,3.0,100,9.0,NaN,NaN,Normal,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-200723-ITS-w-aphid-100D_S277_L001_R1_001.fas...,T-200723-ITS-w-aphid-100D_S277_L001_R2_001.fas...,washed-1-T-200723-ITS-w-aphid-100D,NaN,T-200723-ITS-w-aphid-100D,False,False
236,T-210623-aphid-98D,2023-06-21,timeline,3.0,98,9.0,NaN,NaN,Normal,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-200723-ITS-w-aphid-98D_S276_L001_R1_001.fast...,T-200723-ITS-w-aphid-98D_S276_L001_R2_001.fast...,washed-1-T-200723-ITS-w-aphid-98D,NaN,T-200723-ITS-w-aphid-98D,False,False
235,T-210623-aphid-72D,2023-06-21,timeline,2.0,72,6.0,NaN,NaN,Normal,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-200723-ITS-w-aphid-72D_S275_L001_R1_001.fast...,T-200723-ITS-w-aphid-72D_S275_L001_R2_001.fast...,washed-1-T-200723-ITS-w-aphid-72D,NaN,T-200723-ITS-w-aphid-72D,False,False
234,T-210623-aphid-66D,2023-06-21,timeline,2.0,66,6.0,NaN,NaN,Normal,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-200723-ITS-w-aphid-66D_S274_L001_R1_001.fast...,T-200723-ITS-w-aphid-66D_S274_L001_R2_001.fast...,washed-1-T-200723-ITS-w-aphid-66D,NaN,T-200723-ITS-w-aphid-66D,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-000023-16S-s-P2-pos_S110_L001_R1_001.fastq.gz,T-000023-16S-s-P2-pos_S110_L001_R2_001.fastq.gz,T-2023-16S-s-P2-pos,T-000023-16S-s-P2-pos,T-000023-16S-s-P2-pos,True,False
231,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-000023-ITS-s-P2-pos_S92_L001_R1_001.fastq.gz,T-000023-ITS-s-P2-pos_S92_L001_R2_001.fastq.gz,T-2023-ITS-s-P2-pos,T-000023-ITS-s-P2-pos,T-000023-ITS-s-P2-pos,True,False
110,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-000023-16S-s-P2-neg_S109_L001_R1_001.fastq.gz,T-000023-16S-s-P2-neg_S109_L001_R2_001.fastq.gz,T-2023-16S-s-P2-neg,T-000023-16S-s-P2-neg,T-000023-16S-s-P2-neg,True,False
92,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-000023-16S-s-P1-neg_S92_L001_R1_001.fastq.gz,T-000023-16S-s-P1-neg_S92_L001_R2_001.fastq.gz,T-2023-16S-s-P1-neg,T-000023-16S-s-P1-neg,T-000023-16S-s-P1-neg,True,F

In [23]:
filtered_metadata_shotgun_2023_amplicon_full=metadata_shotgun_2023_amplicon_full[
    metadata_shotgun_2023_amplicon_full.apply(
        lambda row: row['amplicon'] not in row['suggested_correct_seq_sample_ID'], axis=1)]
filtered_metadata_shotgun_2023_amplicon_full

,bio_sample_ID,sample_date,sample_type,block,plot,subplot,ID_on_field,spray,fertilizer,cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT


## Shotgun 2023 

In [24]:
# metadata_shotgun_2023[metadata_shotgun_2023["run_ID"]=="RISO2"]
# metadata_shotgun_2023["run_ID"]=metadata_shotgun_2023["run_ID"].str.replace("RIS2", "RISO2")
print("Number of samples: "  + str(len(metadata_shotgun_2023)))
print("Number of sequence runs: "  + str(metadata_shotgun_2023["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_shotgun_2023["run_ID"].value_counts()


Number of samples: 299
Number of sequence runs: 299

Number of sequence runs per folder: 


run_ID
seq240515_E8KTD    291
seq240222_RQS8X      8
Name: count, dtype: int64

In [25]:
metadata_shotgun_2023_full=metadata_shotgun_2023.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_shotgun_2023_full=metadata_shotgun_2023_full[metadata_shotgun_2023_full["seq_sample_ID"].notna()]

if 'correct_seq_sample_ID' not in metadata_shotgun_2023_full.columns:
    metadata_shotgun_2023_full['correct_seq_sample_ID'] = np.nan
    
metadata_shotgun_2023_full['suggested_correct_seq_sample_ID'] = \
    metadata_shotgun_2023_full['correct_seq_sample_ID'].fillna(metadata_shotgun_2023_full['seq_sample_ID'])
metadata_shotgun_2023_full['suggested_correct_seq_sample_ID'] = metadata_shotgun_2023_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_shotgun_2023_full=metadata_shotgun_2023_full.sort_values("run_ID")

metadata_shotgun_2023_full


,bio_sample_ID,sample_date,sample_type,block,plot,subplot,ID_on_field,spray,fertilizer,cultivar,...,N7_index,index_pair,seq_sample_ID,run_ID,note,correct_seq_sample_ID,Unnamed: 0,dir_name,full_path,suggested_correct_seq_sample_ID
298,T-210623-aphid-107D,2023-06-21,timeline,3.0,107,9.0,NaN,NaN,Normal,sheriff,...,TGCAGCTA,384,T-210623-s-aphid-107D,seq240222_RQS8X,NaN,NaN,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-107D
291,T-210623-aphid-3-61D,2023-06-21,timeline,NaN,3_61,NaN,NaN,NaN,Normal,heerup,...,GTAGAGGA,84,T-210623-s-aphid-3-61D,seq240222_RQS8X,mix_of_sample_3_and_6_pooled_after_centrifugat...,NaN,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-3-61D
292,T-210623-aphid-5-68D,2023-06-21,timeline,NaN,5_68,NaN,NaN,NaN,Normal,sheriff,...,GTAGAGGA,96,T-210623-s-aphid-5-68D,seq240222_RQS8X,mix_of_sample_5_and_68_hole_in_beaker_lost_20ml,NaN,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-5-68D
293,T-210623-aphid-66D,2023-06-21,timeline,2.0,66,6.0,NaN,NaN,Normal,kvium,...,TGCAGCTA,276,T-210623-s-aphid-66D,seq240222_RQS8X,NaN,NaN,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-66D
297,T-210623-aphid-104D,2023-06-21,timeline,3.0,104,9.0,NaN,NaN,Normal,kvium,...,TGCAGCTA,372,T-210623-s-aphid-104D,seq240222_RQS8X,NaN,NaN,2,seq240222_RQS8X,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-104D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,T-130623-80C,2023-06-13,timeline,3.0,80,7.0,NaN,NaN,Half,heerup,...,AAGAGGCA,167,T-130623-s-80C,seq240515_E8KTD,NaN,NaN,21,seq240515_E8KTD,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-130623-s-80C
96,T-130623-75C,2023-06-13,timeline,3.0,75,7.0,NaN,NaN,Half,sheriff,...,AAGAGGCA,155,T-130623-s-75C,seq240515_E8KTD,NaN,NaN,21,seq240515_E8KTD,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-130623-s-75C
95,T-130623-72C,2023-06-13,timeline,2.0,72,6.0,NaN,NaN,Normal,rembrandt,...,AAGAGGCA,143,T-130623-s-72C,seq240515_E8KTD,NaN,NaN,21,seq240515_E8KTD,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-130623-s-72C
93,T-130623-66C,2023-06-13,timeline,2.0,66,6.0,NaN,NaN,Normal,kvium,...,AAGAGGCA,119,T-130623-s-66C,seq240515_E8KTD,NaN,NaN,21,seq240515_E8KTD,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-130623-s-66C


In [26]:
# Iterate over each row and print files
metadata_shotgun_2023_full["forward_fq"]=""
metadata_shotgun_2023_full["reverse_fq"]=""
metadata_shotgun_2023_full["current_seq_sample_ID"]=""
metadata_shotgun_2023_full["corrected_forward_fq"]=""
metadata_shotgun_2023_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_shotgun_2023_full)  # Get the total number of rows

for index, row in metadata_shotgun_2023_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        if len(matching_files) == 2:
            metadata_shotgun_2023_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_shotgun_2023_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_shotgun_2023_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_shotgun_2023_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_shotgun_2023_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2


        if len(matching_files) >2:

            # print(correct_seq_sample_ID)
            
            grouped_files = defaultdict(list)
            
            # Loop over each file and group by the part before R1/R2
            for file in matching_files:
                # Split the file name at R1 or R2
                base_name = file.split("/")[-1].rsplit("_R", 1)[0]  # Split from the last occurrence of "_R"
                grouped_files[base_name].append(file.split("/")[-1])
            
            metadata_shotgun_2023_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", grouped_files[base_name][0].split("/")[-1])[0]
     
            
            # Print the grouped files
            m=0
            for base_name, files in grouped_files.items():      
                if len(files)==2:
            
                    if m==0:
                        # print(grouped_files[base_name][0])
                        metadata_shotgun_2023_full.loc[index, "forward_fq"] = [grouped_files[base_name][0]]
                        metadata_shotgun_2023_full.loc[index, "reverse_fq"] = [grouped_files[base_name][1]]
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2023_full.loc[index, "corrected_forward_fq"] = [correct_seq_sample_ID + ending_R1]
                        metadata_shotgun_2023_full.loc[index, "corrected_reverse_fq"] = [correct_seq_sample_ID + ending_R2]
            
            
                    else:
                        metadata_shotgun_2023_full.loc[index, "forward_fq"].append(grouped_files[base_name][0])
                        metadata_shotgun_2023_full.loc[index, "reverse_fq"].append(grouped_files[base_name][1])
                        ending_R1= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][0].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][0])[1]
                        ending_R2= "_S" + re.search(r"_S(\d+)", grouped_files[base_name][1].split("/")[-1]).group(1) + re.split(r"_S\d+", grouped_files[base_name][1])[1]
                        metadata_shotgun_2023_full.loc[index, "corrected_forward_fq"].append(correct_seq_sample_ID + ending_R1)
                        metadata_shotgun_2023_full.loc[index, "corrected_reverse_fq"].append(correct_seq_sample_ID + ending_R2)
            
                
                    m=m+1
                else:
                    print("ERROR", base_name, files)

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1
    
# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Error occurred for row 200: bio_sample_ID                                                           T-040723-61F
sample_date                                                      2023-07-04 00:00:00
sample_type                                                                 timeline
block                                                                            2.0
plot                                                                              61
subplot                                                                          6.0
ID_on_field                                                                      NaN
spray                                                                            NaN
fertilizer                                                                    Normal
cultivar                                                                      heerup
density                                                                        300.0
x                                    

In [27]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_shotgun_2023_full = metadata_shotgun_2023_full[
    [col for col in metadata_shotgun_2023_full.columns if col not in columns_to_move] + columns_to_move
]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_shotgun_2023_full["SUGGESTED_EQUALS_CORRECT"] = metadata_shotgun_2023_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2023_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_shotgun_2023_full["ALREADY_CORRECT"] = metadata_shotgun_2023_full["suggested_correct_seq_sample_ID"] == metadata_shotgun_2023_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_shotgun_2023_full = metadata_shotgun_2023_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_shotgun_2023_full

,bio_sample_ID,sample_date,sample_type,block,plot,subplot,ID_on_field,spray,fertilizer,cultivar,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
200,T-040723-61F,2023-07-04,timeline,2.0,61,6.0,NaN,NaN,Normal,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,T-040723-s-61F,False,False
199,T-040723-59F,2023-07-04,timeline,2.0,59,5.0,NaN,NaN,Zero,kvium,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,T-040723-s-59F,False,False
198,T-040723-55F,2023-07-04,timeline,2.0,55,5.0,NaN,NaN,Zero,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,T-040723-s-55F,False,False
197,T-040723-53F,2023-07-04,timeline,2.0,53,5.0,NaN,NaN,Zero,sheriff,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,T-040723-s-53F,False,False
196,T-040723-51F,2023-07-04,timeline,2.0,51,5.0,NaN,NaN,Zero,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,T-040723-s-51F,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,T-210623-aphid-72D,2023-06-21,timeline,2.0,72,6.0,NaN,NaN,Normal,rembrandt,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-72D_S156_L001_R1_001.fastq.gz,T-210623-s-aphid-72D_S156_L001_R2_001.fastq.gz,T-210623-s-aphid-72D,NaN,T-210623-s-aphid-72D,False,True
296,T-210623-aphid-100D,2023-06-21,timeline,3.0,100,9.0,NaN,NaN,Normal,heerup,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-210623-s-aphid-100D_S158_L001_R1_001.fastq.gz,T-210623-s-aphid-100D_S158_L001_R2_001.fastq.gz,T-210623-s-aphid-100D,NaN,T-210623-s-aphid-100D,False,True
290,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-000023-s-hug-P4-neg,T-000023-s-hug-P4-neg,True,False
289,Ultrapure-water,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-000023-s-nex15-P3-neg,T-000023-s-nex15-P3-neg,True,False


## WASHED INDIVIDUALLY 2020

In [28]:
metadata_washed_2020.columns = metadata_washed_2020.columns.str.replace('-', '_')
metadata_washed_2020.columns = metadata_washed_2020.columns.str.replace('worrewt', 'correct')

# metadata_washed_2020[metadata_washed_2020["run_ID"]=="RISO2"]
# metadata_washed_2020["run_ID"]=metadata_washed_2020["run_ID"].str.replace("RIS2", "RISO2")
print("Number of samples: "  + str(len(metadata_washed_2020)))
print("Number of sequence runs: "  + str(metadata_washed_2020["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_washed_2020["run_ID"].value_counts()


Number of samples: 108
Number of sequence runs: 108

Number of sequence runs per folder: 


run_ID
seq230714_FWU63    54
seq230710_GW4I8    54
Name: count, dtype: int64

In [29]:
# Merge two DataFrames (metadata_washed_2020 and df_folder_structure) 
# on the columns "run_ID" and "dir_name" respectively, keeping all rows from metadata_washed_2020 
# (left join).

metadata_washed_2020_full = metadata_washed_2020.merge(
    df_folder_structure, left_on="run_ID", right_on="dir_name", how="left"
)

# Filter rows to keep only those where the "seq_sample_ID" column is not NaN.
metadata_washed_2020_full = metadata_washed_2020_full[
    metadata_washed_2020_full["seq_sample_ID"].notna()
]

# Create a new column "suggested_correct_seq_sample_ID". If "correct_seq_sample_ID" is NaN,
# it will use the value from "seq_sample_ID", otherwise, it uses "correct_seq_sample_ID".
metadata_washed_2020_full['suggested_correct_seq_sample_ID'] = \
    metadata_washed_2020_full['correct_seq_sample_ID'].fillna(
        metadata_washed_2020_full['seq_sample_ID']
    )

# Replace underscores with hyphens and remove the prefix "washed-1-" from the values in 
# the "suggested_correct_seq_sample_ID" column.
metadata_washed_2020_full['suggested_correct_seq_sample_ID'] = metadata_washed_2020_full['suggested_correct_seq_sample_ID'].str.replace("_", "-")
metadata_washed_2020_full['suggested_correct_seq_sample_ID'] = metadata_washed_2020_full['suggested_correct_seq_sample_ID'].str.replace("washed-1-", "")

# Sort the DataFrame by the "run_ID" column in ascending order.
metadata_washed_2020_full = metadata_washed_2020_full.sort_values("run_ID")

# End of code, the resulting DataFrame is assigned to metadata_washed_2020_full.
metadata_washed_2020_full


,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,N7_index,index_pair,seq_sample_ID,run_ID,note,correct_seq_sample_ID,Unnamed: 0,dir_name,full_path,suggested_correct_seq_sample_ID
107,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,CTCTCTAC,163,T-2020-ITS-c-pos,seq230710_GW4I8,NaN,T-000020-ITS-w-P1-pos,11,seq230710_GW4I8,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-000020-ITS-w-P1-pos
77,T-030620-w-24,NaN,24.0,2020-06-03,deep,26.0,middle,other,medium,C_10,...,AGGCAGAA,183,T-2020-ITS-c-24,seq230710_GW4I8,NaN,T-030620-ITS-w-24,11,seq230710_GW4I8,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-ITS-w-24
76,T-030620-w-23,NaN,23.0,2020-06-03,deep,25.0,middle,other,medium,C_10,...,AGGCAGAA,171,T-2020-ITS-c-23,seq230710_GW4I8,NaN,T-030620-ITS-w-23,11,seq230710_GW4I8,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-ITS-w-23
75,T-030620-w-22,NaN,22.0,2020-06-03,deep,24.0,middle,other,medium,C_10,...,AGGCAGAA,159,T-2020-ITS-c-22,seq230710_GW4I8,NaN,T-030620-ITS-w-22,11,seq230710_GW4I8,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-ITS-w-22
74,T-030620-w-21,NaN,21.0,2020-06-03,deep,23.0,middle,cross,large,C_10,...,AGGCAGAA,147,T-2020-ITS-c-21,seq230710_GW4I8,NaN,T-030620-ITS-w-21,11,seq230710_GW4I8,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-ITS-w-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30,T-030620-w-31,NaN,31.0,2020-06-03,deep,33.0,middle,other,medium,D_30,...,TCCTGAGC,172,T-2020-16S-c-31,seq230714_FWU63,Alex_big,T-030620-16S-w-31,6,seq230714_FWU63,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-16S-w-31
29,T-030620-w-30,NaN,30.0,2020-06-03,deep,32.0,middle,other,medium,D_30,...,TCCTGAGC,160,T-2020-16S-c-30,seq230714_FWU63,NaN,T-030620-16S-w-30,6,seq230714_FWU63,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-16S-w-30
28,T-030620-w-29,NaN,29.0,2020-06-03,deep,31.0,middle,other,medium,D_30,...,TCCTGAGC,148,T-2020-16S-c-29,seq230714_FWU63,NaN,T-030620-16S-w-29,6,seq230714_FWU63,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-16S-w-29
26,T-030620-w-27,NaN,27.0,2020-06-03,deep,29.0,middle,other,medium,D_30,...,TCCTGAGC,124,T-2020-16S-c-27,seq230714_FWU63,NaN,T-030620-16S-w-27,6,seq230714_FWU63,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-030620-16S-w-27


In [30]:


# Iterate over each row and print files
metadata_washed_2020_full["forward_fq"]=""
metadata_washed_2020_full["reverse_fq"]=""
metadata_washed_2020_full["current_seq_sample_ID"]=""
metadata_washed_2020_full["corrected_forward_fq"]=""
metadata_washed_2020_full["corrected_reverse_fq"]=""

total_rows = len(metadata_washed_2020_full)  # Get the total number of rows
n=0
for index, row in metadata_washed_2020_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID = row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_washed_2020_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_washed_2020_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_washed_2020_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_washed_2020_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_washed_2020_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2

        else:
            prefix2 = prefix.replace("washed-1-", "").replace("-", "_")
            matching_files2 = list_files_with_prefix(folder_path, prefix2)

            if len(matching_files2) == 2:
                metadata_washed_2020_full.loc[index, "forward_fq"] = matching_files2[0]
                metadata_washed_2020_full.loc[index, "reverse_fq"] = matching_files2[1]
                ending_R1= "_S" + re.search(r"_S(\d+)", matching_files2[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[0])[1]
                ending_R2= "_S" + re.search(r"_S(\d+)", matching_files2[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[1])[1]
                metadata_washed_2020_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files2[0].split("/")[-1])[0]
                metadata_washed_2020_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
                metadata_washed_2020_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2
            
        

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")
    
    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")


Processing... 100.00% complete
Processing complete!


In [31]:
# Define a list of columns to be moved to the end
columns_to_move = ["current_seq_sample_ID", "correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]

# Reorder the DataFrame so that the specified columns are moved to the end
metadata_washed_2020_full = metadata_washed_2020_full[
    [col for col in metadata_washed_2020_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_washed_2020_full['forward_fq'] = metadata_washed_2020_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_washed_2020_full['reverse_fq'] = metadata_washed_2020_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_washed_2020_full["SUGGESTED_EQUALS_CORRECT"] = metadata_washed_2020_full["suggested_correct_seq_sample_ID"] == metadata_washed_2020_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_washed_2020_full["ALREADY_CORRECT"] = metadata_washed_2020_full["suggested_correct_seq_sample_ID"] == metadata_washed_2020_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_washed_2020_full = metadata_washed_2020_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])

# Save the resulting DataFrame to a CSV file named 'test_metadata_15_11_2024.csv'
metadata_washed_2020_full


,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,grid_number,grid_point,distance,scheme,sampling_0,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
107,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-ITS-c-pos_S54_L001_R1_001.fastq.gz,T-2020-ITS-c-pos_S54_L001_R2_001.fastq.gz,T-000020-ITS-w-P1-pos_S54_L001_R1_001.fastq.gz,T-000020-ITS-w-P1-pos_S54_L001_R2_001.fastq.gz,T-2020-ITS-c-pos,T-000020-ITS-w-P1-pos,T-000020-ITS-w-P1-pos,True,False
77,T-030620-w-24,NaN,24.0,2020-06-03,deep,26.0,middle,other,medium,C_10,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-ITS-c-24_S24_L001_R1_001.fastq.gz,T-2020-ITS-c-24_S24_L001_R2_001.fastq.gz,T-030620-ITS-w-24_S24_L001_R1_001.fastq.gz,T-030620-ITS-w-24_S24_L001_R2_001.fastq.gz,T-2020-ITS-c-24,T-030620-ITS-w-24,T-030620-ITS-w-24,True,False
76,T-030620-w-23,NaN,23.0,2020-06-03,deep,25.0,middle,other,medium,C_10,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-ITS-c-23_S23_L001_R1_001.fastq.gz,T-2020-ITS-c-23_S23_L001_R2_001.fastq.gz,T-030620-ITS-w-23_S23_L001_R1_001.fastq.gz,T-030620-ITS-w-23_S23_L001_R2_001.fastq.gz,T-2020-ITS-c-23,T-030620-ITS-w-23,T-030620-ITS-w-23,True,False
75,T-030620-w-22,NaN,22.0,2020-06-03,deep,24.0,middle,other,medium,C_10,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-ITS-c-22_S22_L001_R1_001.fastq.gz,T-2020-ITS-c-22_S22_L001_R2_001.fastq.gz,T-030620-ITS-w-22_S22_L001_R1_001.fastq.gz,T-030620-ITS-w-22_S22_L001_R2_001.fastq.gz,T-2020-ITS-c-22,T-030620-ITS-w-22,T-030620-ITS-w-22,True,False
74,T-030620-w-21,NaN,21.0,2020-06-03,deep,23.0,middle,cross,large,C_10,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-ITS-c-21_S21_L001_R1_001.fastq.gz,T-2020-ITS-c-21_S21_L001_R2_001.fastq.gz,T-030620-ITS-w-21_S21_L001_R1_001.fastq.gz,T-030620-ITS-w-21_S21_L001_R2_001.fastq.gz,T-2020-ITS-c-21,T-030620-ITS-w-21,T-030620-ITS-w-21,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30,T-030620-w-31,NaN,31.0,2020-06-03,deep,33.0,middle,other,medium,D_30,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-16S-c-31_S31_L001_R1_001.fastq.gz,T-2020-16S-c-31_S31_L001_R2_001.fastq.gz,T-030620-16S-w-31_S31_L001_R1_001.fastq.gz,T-030620-16S-w-31_S31_L001_R2_001.fastq.gz,T-2020-16S-c-31,T-030620-16S-w-31,T-030620-16S-w-31,True,False
29,T-030620-w-30,NaN,30.0,2020-06-03,deep,32.0,middle,other,medium,D_30,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-16S-c-30_S30_L001_R1_001.fastq.gz,T-2020-16S-c-30_S30_L001_R2_001.fastq.gz,T-030620-16S-w-30_S30_L001_R1_001.fastq.gz,T-030620-16S-w-30_S30_L001_R2_001.fastq.gz,T-2020-16S-c-30,T-030620-16S-w-30,T-030620-16S-w-30,True,False
28,T-030620-w-29,NaN,29.0,2020-06-03,deep,31.0,middle,other,medium,D_30,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-16S-c-29_S29_L001_R1_001.fastq.gz,T-2020-16S-c-29_S29_L001_R2_001.fastq.gz,T-030620-16S-w-29_S29_L001_R1_001.fastq.gz,T-030620-16S-w-29_S29_L001_R2_001.fastq.gz,T-2020-16S-c-29,T-030620-16S-w-29,T-030620-16S-w-29,True,False
26,T-030620-w-27,NaN,27.0,2020-06-03,deep,29.0,middle,other,medium,D_30,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2020-16S-c-27_S27_L001_R1_001.fastq.gz,T-2020-16S-c-27_S27_L001_R2_001.fastq.gz,T-030620-16S-w-27_S27_L001_R1_001.fastq.gz,T-030620-16S-w-27_S27_L001_R2_001.fastq.gz,T-2020-16S-c-27,T-030620-16S-w-27,T-030620-16S-w-27,True,False


## WASHED INDIVIDUALLY 2022

In [32]:

print("Number of samples: "  + str(len(metadata_washed_2022)))
print("Number of sequence runs: "  + str(metadata_washed_2022["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
print(len(metadata_washed_2022["run_ID"]))
metadata_washed_2022["run_ID"].value_counts()


Number of samples: 2252
Number of sequence runs: 2146

Number of sequence runs per folder: 
2252


run_ID
seq240424_T6U6R    344
seq240412_BRLUV    342
seq231206_RISO2    340
seq240209_KSVAZ    337
seq240222_RQS8X    314
seq240508_LNERQ    245
seq231107_RISO1    224
Name: count, dtype: int64

In [33]:
# Merge two DataFrames (metadata_washed_2022 and df_folder_structure) 
# on the columns "run_ID" and "dir_name" respectively, keeping all rows from metadata_washed_2022 
# (left join).
metadata_washed_2022_full = metadata_washed_2022.merge(
    df_folder_structure, left_on="run_ID", right_on="dir_name", how="left"
)

# Filter rows to keep only those where the "seq_sample_ID" column is not NaN.
metadata_washed_2022_full = metadata_washed_2022_full[
    metadata_washed_2022_full["seq_sample_ID"].notna()
]

# Create a new column "suggested_correct_seq_sample_ID". If "correct_seq_sample_ID" is NaN,
# it will use the value from "seq_sample_ID", otherwise, it uses "correct_seq_sample_ID".
metadata_washed_2022_full['suggested_correct_seq_sample_ID'] = \
    metadata_washed_2022_full['correct_seq_sample_ID'].fillna(
        metadata_washed_2022_full['seq_sample_ID']
    )

# Replace underscores with hyphens and remove the prefix "washed-1-" from the values in 
# the "suggested_correct_seq_sample_ID" column.
metadata_washed_2022_full['suggested_correct_seq_sample_ID'] = metadata_washed_2022_full['suggested_correct_seq_sample_ID'].str.replace("_", "-")
metadata_washed_2022_full['suggested_correct_seq_sample_ID'] = metadata_washed_2022_full['suggested_correct_seq_sample_ID'].str.replace("washed-1-", "")

# Sort the DataFrame by the "run_ID" column in ascending order.
metadata_washed_2022_full = metadata_washed_2022_full.sort_values("run_ID")

# End of code, the resulting DataFrame is assigned to metadata_washed_2022_full.
metadata_washed_2022_full


,bio_sample_ID,plot,letter,cultivar,spray,washed_date,DNA_plate_number,DNA_ext_date,Days_between_DNA_x,DNA_conc_individual,...,index_pair,seq_sample_ID,run_number,run_ID,note,correct_seq_sample_ID,Unnamed: 0,dir_name,full_path,suggested_correct_seq_sample_ID
2251,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,...,144.0,washed-1-mix-2023-ITS-w-P14-pos,1.0,seq231107_RISO1,NaN,mix-000022-ITS-w-P14-pos,14.0,seq231107_RISO1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,mix-000022-ITS-w-P14-pos
625,S-240622-w-13-4,13.0,A5,kvium,n,2023-02-24,4.0,2023-02-28,4.0,NaN,...,383.0,washed-1-S-240622-ITS-w-13-4,1.0,seq231107_RISO1,NaN,S-240622-ITS-w-13-4,14.0,seq231107_RISO1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,S-240622-ITS-w-13-4
626,S-240622-w-13-5,13.0,A5,kvium,n,2023-02-24,4.0,2023-02-28,4.0,NaN,...,300.0,washed-1-S-240622-ITS-w-13-5,1.0,seq231107_RISO1,NaN,S-240622-ITS-w-13-5,14.0,seq231107_RISO1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,S-240622-ITS-w-13-5
627,S-240622-w-13-6,13.0,A5,kvium,n,2023-02-24,4.0,2023-02-28,4.0,NaN,...,312.0,washed-1-S-240622-ITS-w-13-6,1.0,seq231107_RISO1,NaN,S-240622-ITS-w-13-6,14.0,seq231107_RISO1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,S-240622-ITS-w-13-6
628,S-240622-w-13-7,13.0,A5,kvium,n,2023-02-24,4.0,2023-02-28,4.0,NaN,...,324.0,washed-1-S-240622-ITS-w-13-7,1.0,seq231107_RISO1,NaN,S-240622-ITS-w-13-7,14.0,seq231107_RISO1,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,S-240622-ITS-w-13-7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2058,R-210622-w-21-1,21.0,B7,Heerup,n,2023-03-10,13.0,2023-03-28,18.0,NaN,...,103.0,R-210622-16S-w-21-1,4.0,seq240508_LNERQ,NaN,NaN,10.0,seq240508_LNERQ,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-w-21-1
2057,R-210622-w-20-7,20.0,A7,Heerup,y,2023-03-10,13.0,2023-03-28,18.0,NaN,...,186.0,R-210622-16S-w-20-7,4.0,seq240508_LNERQ,NaN,NaN,10.0,seq240508_LNERQ,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-w-20-7
2056,R-210622-w-20-6,20.0,A7,Heerup,y,2023-03-10,13.0,2023-03-28,18.0,NaN,...,174.0,R-210622-16S-w-20-6,4.0,seq240508_LNERQ,DNA_input_below_100pg,NaN,10.0,seq240508_LNERQ,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-w-20-6
2062,R-210622-w-21-5,21.0,B7,Heerup,n,2023-03-10,13.0,2023-03-28,18.0,NaN,...,151.0,R-210622-16S-w-21-5,4.0,seq240508_LNERQ,NaN,NaN,10.0,seq240508_LNERQ,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-w-21-5


In [34]:
# Iterate over each row and print files
metadata_washed_2022_full["forward_fq"]=""
metadata_washed_2022_full["reverse_fq"]=""
metadata_washed_2022_full["current_seq_sample_ID"]=""
metadata_washed_2022_full["corrected_forward_fq"]=""
metadata_washed_2022_full["corrected_reverse_fq"]=""

total_rows = len(metadata_washed_2022_full)  # Get the total number of rows
n=0
for index, row in metadata_washed_2022_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID = row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_washed_2022_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_washed_2022_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_washed_2022_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_washed_2022_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_washed_2022_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2

        else:
            prefix2 = prefix.replace("washed-1-", "").replace("-", "_")
            matching_files2 = list_files_with_prefix(folder_path, prefix2)

            if len(matching_files2) == 2:
                metadata_washed_2022_full.loc[index, "forward_fq"] = matching_files2[0]
                metadata_washed_2022_full.loc[index, "reverse_fq"] = matching_files2[1]
                ending_R1= "_S" + re.search(r"_S(\d+)", matching_files2[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[0])[1]
                ending_R2= "_S" + re.search(r"_S(\d+)", matching_files2[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[1])[1]
                metadata_washed_2022_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files2[0].split("/")[-1])[0]
                metadata_washed_2022_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
                metadata_washed_2022_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2
            
        

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")
    
    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")


Processing... 100.00% complete
Processing complete!


In [35]:
# Define a list of columns to be moved to the end
columns_to_move = ["current_seq_sample_ID", "correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]

# Reorder the DataFrame so that the specified columns are moved to the end
metadata_washed_2022_full = metadata_washed_2022_full[
    [col for col in metadata_washed_2022_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_washed_2022_full['forward_fq'] = metadata_washed_2022_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_washed_2022_full['reverse_fq'] = metadata_washed_2022_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_washed_2022_full["SUGGESTED_EQUALS_CORRECT"] = metadata_washed_2022_full["suggested_correct_seq_sample_ID"] == metadata_washed_2022_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_washed_2022_full["ALREADY_CORRECT"] = metadata_washed_2022_full["suggested_correct_seq_sample_ID"] == metadata_washed_2022_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_washed_2022_full = metadata_washed_2022_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])

# Save the resulting DataFrame to a CSV file named 'test_metadata_15_11_2024.csv'
metadata_washed_2022_full


,bio_sample_ID,plot,letter,cultivar,spray,washed_date,DNA_plate_number,DNA_ext_date,Days_between_DNA_x,DNA_conc_individual,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
720,S-240622-w-15-3,15.0,A16,rembrandt,n,2023-02-24,5.0,2023-02-28,4.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,Washed_S-240622-ITS-w-15-3_S9_L001_R1_001.fast...,Washed_S-240622-ITS-w-15-3_S9_L001_R2_001.fast...,S-240622-ITS-w-15-3_S9_L001_R1_001.fastq.gz,S-240622-ITS-w-15-3_S9_L001_R2_001.fastq.gz,Washed_S-240622-ITS-w-15-3,NaN,S-240622-ITS-w-15-3,False,False
719,S-240622-w-15-2,15.0,A16,rembrandt,n,2023-02-24,5.0,2023-02-28,4.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,Washed_S-240622-ITS-w-15-2_S8_L001_R1_001.fast...,Washed_S-240622-ITS-w-15-2_S8_L001_R2_001.fast...,S-240622-ITS-w-15-2_S8_L001_R1_001.fastq.gz,S-240622-ITS-w-15-2_S8_L001_R2_001.fastq.gz,Washed_S-240622-ITS-w-15-2,NaN,S-240622-ITS-w-15-2,False,False
718,S-240622-w-15-1,15.0,A16,rembrandt,n,2023-02-24,5.0,2023-02-28,4.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,Washed_S-240622-ITS-w-15-1_S7_L001_R1_001.fast...,Washed_S-240622-ITS-w-15-1_S7_L001_R2_001.fast...,S-240622-ITS-w-15-1_S7_L001_R1_001.fastq.gz,S-240622-ITS-w-15-1_S7_L001_R2_001.fastq.gz,Washed_S-240622-ITS-w-15-1,NaN,S-240622-ITS-w-15-1,False,False
717,S-240622-w-14-7,14.0,A7,heerup,n,2023-02-24,5.0,2023-02-28,4.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,Washed_S-240622-ITS-w-14-7_S6_L001_R1_001.fast...,Washed_S-240622-ITS-w-14-7_S6_L001_R2_001.fast...,S-240622-ITS-w-14-7_S6_L001_R1_001.fastq.gz,S-240622-ITS-w-14-7_S6_L001_R2_001.fastq.gz,Washed_S-240622-ITS-w-14-7,NaN,S-240622-ITS-w-14-7,False,False
716,S-240622-w-14-6,14.0,A7,heerup,n,2023-02-24,5.0,2023-02-28,4.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,Washed_S-240622-ITS-w-14-6_S5_L001_R1_001.fast...,Washed_S-240622-ITS-w-14-6_S5_L001_R2_001.fast...,S-240622-ITS-w-14-6_S5_L001_R1_001.fastq.gz,S-240622-ITS-w-14-6_S5_L001_R2_001.fastq.gz,Washed_S-240622-ITS-w-14-6,NaN,S-240622-ITS-w-14-6,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1571,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,L-2022-16S-w-P10-pos_S86_L001_R1_001.fastq.gz,L-2022-16S-w-P10-pos_S86_L001_R2_001.fastq.gz,L-000022-16S-w-P10-pos_S86_L001_R1_001.fastq.gz,L-000022-16S-w-P10-pos_S86_L001_R2_001.fastq.gz,L-2022-16S-w-P10-pos,L-000022-16S-w-P10-pos,L-000022-16S-w-P10-pos,True,False
2089,Ultrapure-water,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-2022-16S-w-P13-neg_S174_L001_R1_001.fastq.gz,R-2022-16S-w-P13-neg_S174_L001_R2_001.fastq.gz,R-000022-16S-w-P13-neg_S174_L001_R1_001.fastq.gz,R-000022-16S-w-P13-neg_S174_L001_R2_001.fastq.gz,R-2022-16S-w-P13-neg,R-000022-16S-w-P13-neg,R-000022-16S-w-P13-neg,True,False
2090,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-2022-16S-w-P13-pos_S175_L001_R1_001.fastq.gz,R-2022-16S-w-P13-pos_S175_L001_R2_001.fastq.gz,R-000022-16S-w-P13-pos_S175_L001_R1_001.fastq.gz,R-000022-16S-w-P13-pos_S175_L001_R2_001.fastq.gz,R-2022-16S-w-P13-pos,R-000022-16S-w-P13-pos,R-000022-16S-w-P13-pos,True,False
1455,A-230622-w-22-1,22.0,A5,kvuim,n,2023-08-01,9.0,2023-08-08,7.0,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,A-230622-ITS-w-22-1_S56_L001_R1_001.fastq.gz,A-230622-ITS-w-22-1_S56_L001_R2_001.fastq.gz,A-230622-ITS-w-22-1_S56_L001_R1_001.fastq.gz,A-230622-ITS-w-22-1_S56_L001_R2_001.fastq.gz,A-230622-ITS-w-22-1,A-230622-ITS-w-22-1,A-230622-ITS-w-22-1,True,True


## CRUSHED 2021

In [36]:

print("Number of samples: "  + str(len(metadata_crushed_2021)))
print("Number of sequence runs: "  + str(metadata_crushed_2021["run_ID"].value_counts().sum()))
print()

print("Number of sequence runs per folder: ")
metadata_crushed_2021["run_ID"].value_counts()


NameError: name 'metadata_crushed_2021' is not defined

In [ ]:
metadata_crushed_2021_full=metadata_crushed_2021.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="right")
metadata_crushed_2021_full
metadata_crushed_2021_full=metadata_crushed_2021_full[metadata_crushed_2021_full["seq_sample_ID"].notna()]
metadata_crushed_2021_full['suggested_correct_seq_sample_ID'] = \
    metadata_crushed_2021_full['correct_seq_sample_ID'].fillna(metadata_crushed_2021_full['seq_sample_ID'])
metadata_crushed_2021_full['suggested_correct_seq_sample_ID'] = metadata_crushed_2021_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "").str.replace("16s", "16S")
metadata_crushed_2021_full=metadata_crushed_2021_full.sort_values("run_ID")

# ###### REMOVE WHEN RISO1 IS FIXED!!!!!
# metadata_crushed_2021_full=metadata_crushed_2021_full[metadata_crushed_2021_full["run_ID"]!="seq231107_RISO1"]
# metadata_crushed_2021_full



In [ ]:
# Iterate over each row and print files
metadata_crushed_2021_full["forward_fq"]=""
metadata_crushed_2021_full["reverse_fq"]=""
metadata_crushed_2021_full["current_seq_sample_ID"]=""
metadata_crushed_2021_full["corrected_forward_fq"]=""
metadata_crushed_2021_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_crushed_2021_full)  # Get the total number of rows

for index, row in metadata_crushed_2021_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_crushed_2021_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_crushed_2021_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_crushed_2021_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_crushed_2021_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_crushed_2021_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2
        else:
            # prefix2=prefix.replace("crushed-1-", "").replace("-", "_")
            prefix2=prefix.replace("washed-1-", "")#.replace("-", "_")
            matching_files2 = list_files_with_prefix(folder_path, prefix2)
    
            if len(matching_files2) == 2:
                metadata_crushed_2021_full.loc[index, "forward_fq"] = matching_files2[0]
                metadata_crushed_2021_full.loc[index, "reverse_fq"] = matching_files2[1]
                ending_R1= "_S" + re.search(r"_S(\d+)", matching_files2[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[0])[1]
                ending_R2= "_S" + re.search(r"_S(\d+)", matching_files2[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files2[1])[1]
                metadata_crushed_2021_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files2[0].split("/")[-1])[0]
                metadata_crushed_2021_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
                metadata_crushed_2021_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2
            else:
                print(f"Files in {folder_path} containing {prefix2}:")
            
            # print(correct_seq_sample_ID)
            # print(prefix)
            # print()

        #         print(f" - {prefix}")
        #         print(f" - {matching_files}")

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-128_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-127_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-126_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-125_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-123_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-122_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbiomics/Metagenomics/sequencing_runs/seq231107_RISO1 containing T-2021-16s-c-129_:
Files in /home/jovyan/work/CCRP/MATRIX/RawData/Microbio

In [ ]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_crushed_2021_full = metadata_crushed_2021_full[
    [col for col in metadata_crushed_2021_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_crushed_2021_full['forward_fq'] = metadata_crushed_2021_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_crushed_2021_full['reverse_fq'] = metadata_crushed_2021_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_crushed_2021_full["SUGGESTED_EQUALS_CORRECT"] = metadata_crushed_2021_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2021_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_crushed_2021_full["ALREADY_CORRECT"] = metadata_crushed_2021_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2021_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_crushed_2021_full = metadata_crushed_2021_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_crushed_2021_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
311,T-150621-c-mystery,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,NaN,washed-1-T-2021-16S-c-mystery,False,False
184,T-150621-c-118,NaN,118.0,2021-06-15,deep,4.0,118.0,10.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2021-ITS-c-118_S19_L001_R1_001.fastq.gz,T-2021-ITS-c-118_S19_L001_R2_001.fastq.gz,T-150621-ITS-c-118_S19_L001_R1_001.fastq.gz,T-150621-ITS-c-118_S19_L001_R2_001.fastq.gz,T-2021-ITS-c-118,T-150621-ITS-c-118,T-150621-ITS-c-118,True,False
212,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2021-P1-ITS-c-pos_S147_L001_R1_001.fastq.gz,T-2021-P1-ITS-c-pos_S147_L001_R2_001.fastq.gz,T-000021-ITS-c-P1-pos_S147_L001_R1_001.fastq.gz,T-000021-ITS-c-P1-pos_S147_L001_R2_001.fastq.gz,T-2021-P1-ITS-c-pos,T-000021-ITS-c-P1-pos,T-000021-ITS-c-P1-pos,True,False
211,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2021-P1-ITS-c-neg_S148_L001_R1_001.fastq.gz,T-2021-P1-ITS-c-neg_S148_L001_R2_001.fastq.gz,T-000021-ITS-c-P1-neg_S148_L001_R1_001.fastq.gz,T-000021-ITS-c-P1-neg_S148_L001_R2_001.fastq.gz,T-2021-P1-ITS-c-neg,T-000021-ITS-c-P1-neg,T-000021-ITS-c-P1-neg,True,False
210,T-150621-c-144,NaN,144.0,2021-06-15,deep,4.0,144.0,12.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2021-ITS-c-144_S45_L001_R1_001.fastq.gz,T-2021-ITS-c-144_S45_L001_R2_001.fastq.gz,T-150621-ITS-c-144_S45_L001_R1_001.fastq.gz,T-150621-ITS-c-144_S45_L001_R2_001.fastq.gz,T-2021-ITS-c-144,T-150621-ITS-c-144,T-150621-ITS-c-144,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,T-150621-c-113,NaN,113.0,2021-06-15,deep,4.0,113.0,10.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-150621-16S-c-113,T-150621-16S-c-113,True,False
331,T-150621-c-114,NaN,114.0,2021-06-15,deep,4.0,114.0,10.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-150621-16S-c-114,T-150621-16S-c-114,True,False
332,T-150621-c-115,NaN,115.0,2021-06-15,deep,4.0,115.0,10.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-150621-16S-c-115,T-150621-16S-c-115,True,False
321,T-150621-c-104,NaN,104.0,2021-06-15,deep,3.0,104.0,9.0,NaN,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,,,,,,T-150621-16S-c-104,T-150621-16S-c-104,True,False


## CRUSHED 2022

In [ ]:
# metadata_crushed_2022[metadata_crushed_2022["run_ID"]=="RISO2"]
# metadata_crushed_2022["run_ID"]=metadata_crushed_2022["run_ID"].str.replace("RIS2", "RISO2")
print("Number of samples: "  + str(len(metadata_crushed_2022)))
print("Number of sequence runs: "  + str(metadata_crushed_2022["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_crushed_2022["run_ID"].value_counts()


Number of samples: 737
Number of sequence runs: 731

Number of sequence runs per folder: 


run_ID
seq230707_DWDQH    367
seq230526_JRV78    364
Name: count, dtype: int64

In [ ]:
metadata_crushed_2022_full=metadata_crushed_2022.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_crushed_2022_full=metadata_crushed_2022_full[metadata_crushed_2022_full["seq_sample_ID"].notna()]
metadata_crushed_2022_full['suggested_correct_seq_sample_ID'] = \
    metadata_crushed_2022_full['correct_seq_sample_ID'].fillna(metadata_crushed_2022_full['seq_sample_ID'])
metadata_crushed_2022_full['suggested_correct_seq_sample_ID'] = metadata_crushed_2022_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "")
metadata_crushed_2022_full=metadata_crushed_2022_full.sort_values("run_ID")
metadata_crushed_2022_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,N7_index,index_pair,seq_sample_ID,run_ID,note,correct_seq_sample_ID,Unnamed: 0,dir_name,full_path,suggested_correct_seq_sample_ID
368,R-160622-c-1,NaN,1.0,2022-06-16,timeline,1.0,1.0,NaN,A16,n,...,TAAGGCGA,1.0,R-160622-ITS-c-1,seq230526_JRV78,Alex_may__have_thawed_before_freezedrying,NaN,15.0,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-160622-ITS-c-1
488,H-200622-c-17-7,NaN,17.0,2022-06-20,deep,3.0,8.0,NaN,A7,n,...,CGGAGCCT,221.0,H-200622-ITS-c-17-7,seq230526_JRV78,NaN,NaN,15.0,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-17-7
489,H-200622-c-18-1,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,CGGAGCCT,233.0,H-200622-ITS-c-18-1,seq230526_JRV78,Alex_May_have_thawed_before_freezedrying,NaN,15.0,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-1
490,H-200622-c-18-2,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,CGGAGCCT,245.0,H-200622-ITS-c-18-2,seq230526_JRV78,NaN,NaN,15.0,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-2
491,H-200622-c-18-3,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,CGGAGCCT,257.0,H-200622-ITS-c-18-3,seq230526_JRV78,NaN,NaN,15.0,seq230526_JRV78,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,R-210622-c-43-6,NaN,43.0,2022-06-21,deep,13.0,19.0,NaN,A16,n,...,GCTACGCT,141.0,R-210622-16S-c-43-6,seq230707_DWDQH,NaN,NaN,0.0,seq230707_DWDQH,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-c-43-6
250,R-210622-c-43-7,NaN,43.0,2022-06-21,deep,13.0,19.0,NaN,A16,n,...,GCTACGCT,153.0,R-210622-16S-c-43-7,seq230707_DWDQH,NaN,NaN,0.0,seq230707_DWDQH,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-c-43-7
251,R-210622-c-44-1,NaN,44.0,2022-06-21,deep,13.0,20.0,NaN,A7,n,...,GCTACGCT,165.0,R-210622-16S-c-44-1,seq230707_DWDQH,NaN,NaN,0.0,seq230707_DWDQH,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-c-44-1
253,R-210622-c-44-3,NaN,44.0,2022-06-21,deep,13.0,20.0,NaN,A7,n,...,GCTACGCT,189.0,R-210622-16S-c-44-3,seq230707_DWDQH,NaN,NaN,0.0,seq230707_DWDQH,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-210622-16S-c-44-3


In [ ]:
# Iterate over each row and print files
metadata_crushed_2022_full["forward_fq"]=""
metadata_crushed_2022_full["reverse_fq"]=""
metadata_crushed_2022_full["current_seq_sample_ID"]=""
metadata_crushed_2022_full["corrected_forward_fq"]=""
metadata_crushed_2022_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_crushed_2022_full)  # Get the total number of rows

for index, row in metadata_crushed_2022_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_crushed_2022_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_crushed_2022_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_crushed_2022_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_crushed_2022_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_crushed_2022_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2
    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Processing... 100.00% complete
Processing complete!


In [ ]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_crushed_2022_full = metadata_crushed_2022_full[
    [col for col in metadata_crushed_2022_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_crushed_2022_full['forward_fq'] = metadata_crushed_2022_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_crushed_2022_full['reverse_fq'] = metadata_crushed_2022_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_crushed_2022_full["SUGGESTED_EQUALS_CORRECT"] = metadata_crushed_2022_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2022_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_crushed_2022_full["ALREADY_CORRECT"] = metadata_crushed_2022_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2022_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_crushed_2022_full = metadata_crushed_2022_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_crushed_2022_full


,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,ID_on_field,spray_fungicide,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
368,R-160622-c-1,NaN,1.0,2022-06-16,timeline,1.0,1.0,NaN,A16,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-160622-ITS-c-1_S1_L001_R1_001.fastq.gz,R-160622-ITS-c-1_S1_L001_R2_001.fastq.gz,R-160622-ITS-c-1_S1_L001_R1_001.fastq.gz,R-160622-ITS-c-1_S1_L001_R2_001.fastq.gz,R-160622-ITS-c-1,NaN,R-160622-ITS-c-1,False,True
488,H-200622-c-17-7,NaN,17.0,2022-06-20,deep,3.0,8.0,NaN,A7,n,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-17-7_S121_L001_R1_001.fastq.gz,H-200622-ITS-c-17-7_S121_L001_R2_001.fastq.gz,H-200622-ITS-c-17-7_S121_L001_R1_001.fastq.gz,H-200622-ITS-c-17-7_S121_L001_R2_001.fastq.gz,H-200622-ITS-c-17-7,NaN,H-200622-ITS-c-17-7,False,True
489,H-200622-c-18-1,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-1_S122_L001_R1_001.fastq.gz,H-200622-ITS-c-18-1_S122_L001_R2_001.fastq.gz,H-200622-ITS-c-18-1_S122_L001_R1_001.fastq.gz,H-200622-ITS-c-18-1_S122_L001_R2_001.fastq.gz,H-200622-ITS-c-18-1,NaN,H-200622-ITS-c-18-1,False,True
490,H-200622-c-18-2,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-2_S123_L001_R1_001.fastq.gz,H-200622-ITS-c-18-2_S123_L001_R2_001.fastq.gz,H-200622-ITS-c-18-2_S123_L001_R1_001.fastq.gz,H-200622-ITS-c-18-2_S123_L001_R2_001.fastq.gz,H-200622-ITS-c-18-2,NaN,H-200622-ITS-c-18-2,False,True
491,H-200622-c-18-3,NaN,18.0,2022-06-20,deep,3.0,9.0,NaN,B5,y,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,H-200622-ITS-c-18-3_S124_L001_R1_001.fastq.gz,H-200622-ITS-c-18-3_S124_L001_R2_001.fastq.gz,H-200622-ITS-c-18-3_S124_L001_R1_001.fastq.gz,H-200622-ITS-c-18-3_S124_L001_R2_001.fastq.gz,H-200622-ITS-c-18-3,NaN,H-200622-ITS-c-18-3,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-P1-16S-c-pos_S88_L001_R1_001.fastq.gz,R-P1-16S-c-pos_S88_L001_R2_001.fastq.gz,R-000022-16S-c-P1-pos_S88_L001_R1_001.fastq.gz,R-000022-16S-c-P1-pos_S88_L001_R2_001.fastq.gz,R-P1-16S-c-pos,R-000022-16S-c-P1-pos,R-000022-16S-c-P1-pos,True,False
85,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-P1-16S-c-neg_S89_L001_R1_001.fastq.gz,R-P1-16S-c-neg_S89_L001_R2_001.fastq.gz,R-000022-16S-c-P1-neg_S89_L001_R1_001.fastq.gz,R-000022-16S-c-P1-neg_S89_L001_R2_001.fastq.gz,R-P1-16S-c-neg,R-000022-16S-c-P1-neg,R-000022-16S-c-P1-neg,True,False
367,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-P4-16S-c-neg_S187_L001_R1_001.fastq.gz,R-P4-16S-c-neg_S187_L001_R2_001.fastq.gz,R-000022-16S-c-P4-neg_S187_L001_R1_001.fastq.gz,R-000022-16S-c-P4-neg_S187_L001_R2_001.fastq.gz,R-P4-16S-c-neg,R-000022-16S-c-P4-neg,R-000022-16S-c-P4-neg,True,False
277,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,R-P3-16S-c-pos_S97_L001_R1_001.fastq.gz,R-P3-16S-c-pos_S97_L001_R2_001.fastq.gz,R-000022-16S-c-P3-pos_S97_L001_R1_001.fastq.gz,R-000022-16S-c-P3-pos_S97_L001_R2_001.fastq.gz,R-P3-16S-c-pos,R-000022-16S-c-P3-pos,R-000022-16S-c-P3-pos,True,False


## CRUSHED 2023 16S

In [ ]:
# metadata_crushed_2023_16S[metadata_crushed_2023_16S["run_ID"]=="RISO2"]
# metadata_crushed_2023_16S["run_ID"]=metadata_crushed_2023_16S["run_ID"].str.replace("RIS2", "RISO2")
print("Number of samples: "  + str(len(metadata_crushed_2023_16S)))
print("Number of sequence runs: "  + str(metadata_crushed_2023_16S["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_crushed_2023_16S["run_ID"].value_counts()


Number of samples: 626
Number of sequence runs: 583

Number of sequence runs per folder: 


run_ID
seq240820_C6WGP    189
seq240710_DR8PG    161
seq240826_X6DU5    140
seq240719_NR8NI     93
Name: count, dtype: int64

In [ ]:
metadata_crushed_2023_16S_full=metadata_crushed_2023_16S.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_crushed_2023_16S_full=metadata_crushed_2023_16S_full[metadata_crushed_2023_16S_full["seq_sample_ID"].notna()]
metadata_crushed_2023_16S_full['suggested_correct_seq_sample_ID'] = \
    metadata_crushed_2023_16S_full['correct_seq_sample_ID'].fillna(metadata_crushed_2023_16S_full['seq_sample_ID'])
metadata_crushed_2023_16S_full['suggested_correct_seq_sample_ID'] = metadata_crushed_2023_16S_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "").str.replace("16s", "16S")
metadata_crushed_2023_16S_full=metadata_crushed_2023_16S_full.sort_values("run_ID")
metadata_crushed_2023_16S_full = metadata_crushed_2023_16S_full.dropna(axis=1, how='all')



In [ ]:
# Iterate over each row and print files
metadata_crushed_2023_16S_full["forward_fq"]=""
metadata_crushed_2023_16S_full["reverse_fq"]=""
metadata_crushed_2023_16S_full["current_seq_sample_ID"]=""
metadata_crushed_2023_16S_full["corrected_forward_fq"]=""
metadata_crushed_2023_16S_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_crushed_2023_16S_full)  # Get the total number of rows

for index, row in metadata_crushed_2023_16S_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_crushed_2023_16S_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_crushed_2023_16S_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_crushed_2023_16S_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_crushed_2023_16S_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_crushed_2023_16S_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2


    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Processing... 100.00% complete
Processing complete!


In [ ]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_crushed_2023_16S_full = metadata_crushed_2023_16S_full[
    [col for col in metadata_crushed_2023_16S_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_crushed_2023_16S_full['forward_fq'] = metadata_crushed_2023_16S_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_crushed_2023_16S_full['reverse_fq'] = metadata_crushed_2023_16S_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_crushed_2023_16S_full["SUGGESTED_EQUALS_CORRECT"] = metadata_crushed_2023_16S_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2023_16S_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_crushed_2023_16S_full["ALREADY_CORRECT"] = metadata_crushed_2023_16S_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2023_16S_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_crushed_2023_16S_full = metadata_crushed_2023_16S_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_crushed_2023_16S_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,spray_fungicide,fertiliser,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
308,T-110723-c-41F-1,1037.0,41.0,2023-07-11,timeline,2.0,41.0,4.0,n,Half,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-110723-16S-c-41F-1_S11_L001_R1_001.fastq.gz,T-110723-16S-c-41F-1_S11_L001_R2_001.fastq.gz,T-110723-16S-c-41F-1_S11_L001_R1_001.fastq.gz,T-110723-16S-c-41F-1_S11_L001_R2_001.fastq.gz,T-110723-16S-c-41F-1,NaN,T-110723-16S-c-41F-1,False,True
262,T-040723-c-22E-1,845.0,22.0,2023-07-04,timeline,1.0,22.0,2.0,n,Half,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-040723-16S-c-22E-1_S153_L001_R1_001.fastq.gz,T-040723-16S-c-22E-1_S153_L001_R2_001.fastq.gz,T-040723-16S-c-22E-1_S153_L001_R1_001.fastq.gz,T-040723-16S-c-22E-1_S153_L001_R2_001.fastq.gz,T-040723-16S-c-22E-1,NaN,T-040723-16S-c-22E-1,False,True
261,T-040723-c-18E-1,849.0,18.0,2023-07-04,timeline,1.0,18.0,2.0,n,Half,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-040723-16S-c-18E-1_S152_L001_R1_001.fastq.gz,T-040723-16S-c-18E-1_S152_L001_R2_001.fastq.gz,T-040723-16S-c-18E-1_S152_L001_R1_001.fastq.gz,T-040723-16S-c-18E-1_S152_L001_R2_001.fastq.gz,T-040723-16S-c-18E-1,NaN,T-040723-16S-c-18E-1,False,True
260,T-040723-c-17E-1,853.0,17.0,2023-07-04,timeline,1.0,17.0,2.0,n,Half,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-040723-16S-c-17E-1_S151_L001_R1_001.fastq.gz,T-040723-16S-c-17E-1_S151_L001_R2_001.fastq.gz,T-040723-16S-c-17E-1_S151_L001_R1_001.fastq.gz,T-040723-16S-c-17E-1_S151_L001_R2_001.fastq.gz,T-040723-16S-c-17E-1,NaN,T-040723-16S-c-17E-1,False,True
258,T-040723-c-9E-1,801.0,9.0,2023-07-04,timeline,1.0,9.0,1.0,n,Normal,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-040723-16S-c-9E-1_S149_L001_R1_001.fastq.gz,T-040723-16S-c-9E-1_S149_L001_R2_001.fastq.gz,T-040723-16S-c-9E-1_S149_L001_R1_001.fastq.gz,T-040723-16S-c-9E-1_S149_L001_R2_001.fastq.gz,T-040723-16S-c-9E-1,NaN,T-040723-16S-c-9E-1,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-16S-c-P6-pos_S150_L001_R1_001.fastq.gz,T-2023-16S-c-P6-pos_S150_L001_R2_001.fastq.gz,T-000023-16S-c-P6-pos_S150_L001_R1_001.fastq.gz,T-000023-16S-c-P6-pos_S150_L001_R2_001.fastq.gz,T-2023-16S-c-P6-pos,T-000023-16S-c-P6-pos,T-000023-16S-c-P6-pos,True,False
93,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-16S-c-P1-pos_S2_L001_R1_001.fastq.gz,T-2023-16S-c-P1-pos_S2_L001_R2_001.fastq.gz,T-000023-16S-c-P1-pos_S2_L001_R1_001.fastq.gz,T-000023-16S-c-P1-pos_S2_L001_R2_001.fastq.gz,T-2023-16S-c-P1-pos,T-000023-16S-c-P1-pos,T-000023-16S-c-P1-pos,True,False
624,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-16S-c-P7-neg_S141_L001_R1_001.fastq.gz,T-2023-16S-c-P7-neg_S141_L001_R2_001.fastq.gz,T-000023-16S-c-P7-neg_S141_L001_R1_001.fastq.gz,T-000023-16S-c-P7-neg_S141_L001_R2_001.fastq.gz,T-2023-16S-c-P7-neg,T-000023-16S-c-P7-neg,T-000023-16S-c-P7-neg,True,False
92,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-16S-c-P1-neg_S1_L001_R1_001.fastq.gz,T-2023-16S-c-P1-neg_S1_L001_R2_001.fastq.gz,T-000023-16S-c-P1-neg_S1_L001_R1_001.fastq.gz,T-000023-16S-c-P1-neg_S1_L001_R2_001.fastq.gz,T-2023-16S-c-P1-neg,T-000023-16S-c-P1-neg,T-000023-16S-c-P1-neg,True,False


In [ ]:
filtered_metadata_crushed_2023_16S_full=metadata_crushed_2023_16S_full[
    metadata_crushed_2023_16S_full.apply(
        lambda row: row['amplicon'] not in row['suggested_correct_seq_sample_ID'], axis=1)]
filtered_metadata_crushed_2023_16S_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,spray_fungicide,fertiliser,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT


## CRUSHED 2023 ITS

In [ ]:
# metadata_crushed_2023_ITS[metadata_crushed_2023_ITS["run_ID"]=="RISO2"]
# metadata_crushed_2023_ITS["run_ID"]=metadata_crushed_2023_ITS["run_ID"].str.replace("RIS2", "RISO2")
print("Number of samples: "  + str(len(metadata_crushed_2023_ITS)))
print("Number of sequence runs: "  + str(metadata_crushed_2023_ITS["run_ID"].value_counts().sum()))
print()
print("Number of sequence runs per folder: ")
metadata_crushed_2023_ITS["run_ID"].value_counts()


Number of samples: 628
Number of sequence runs: 585

Number of sequence runs per folder: 


run_ID
seq240826_X6DU5    188
seq240719_NR8NI    162
seq240820_C6WGP    142
seq240710_DR8PG     93
Name: count, dtype: int64

In [ ]:
metadata_crushed_2023_ITS_full=metadata_crushed_2023_ITS.merge(df_folder_structure, left_on="run_ID", right_on="dir_name", how="left")
metadata_crushed_2023_ITS_full=metadata_crushed_2023_ITS_full[metadata_crushed_2023_ITS_full["seq_sample_ID"].notna()]
metadata_crushed_2023_ITS_full['suggested_correct_seq_sample_ID'] = \
    metadata_crushed_2023_ITS_full['correct_seq_sample_ID'].fillna(metadata_crushed_2023_ITS_full['seq_sample_ID'])
metadata_crushed_2023_ITS_full['suggested_correct_seq_sample_ID'] = metadata_crushed_2023_ITS_full['suggested_correct_seq_sample_ID'].str.replace("_", "-").str.replace("crushed-1-", "").str.replace("16s", "16S")
metadata_crushed_2023_ITS_full=metadata_crushed_2023_ITS_full.sort_values("run_ID")
metadata_crushed_2023_ITS_full = metadata_crushed_2023_ITS_full.dropna(axis=1, how='all')



In [ ]:


# Iterate over each row and print files
metadata_crushed_2023_ITS_full["forward_fq"]=""
metadata_crushed_2023_ITS_full["reverse_fq"]=""
metadata_crushed_2023_ITS_full["current_seq_sample_ID"]=""
metadata_crushed_2023_ITS_full["corrected_forward_fq"]=""
metadata_crushed_2023_ITS_full["corrected_reverse_fq"]=""
n=0
total_rows = len(metadata_crushed_2023_ITS_full)  # Get the total number of rows

for index, row in metadata_crushed_2023_ITS_full.iterrows():
    try:
        folder_path = row['full_path']  # Get the folder path
        prefix = row['seq_sample_ID'] + "_"  # Get the prefix from the column
        matching_files = list_files_with_prefix(folder_path, prefix)
        correct_seq_sample_ID= row['suggested_correct_seq_sample_ID']

        # Check if the number of matching files is not equal to 2
        if len(matching_files) == 2:
            metadata_crushed_2023_ITS_full.loc[index, "forward_fq"] = matching_files[0]
            metadata_crushed_2023_ITS_full.loc[index, "reverse_fq"] = matching_files[1]
            ending_R1= "_S" + re.search(r"_S(\d+)", matching_files[0].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[0])[1]
            ending_R2= "_S" + re.search(r"_S(\d+)", matching_files[1].split("/")[-1]).group(1) + re.split(r"_S\d+", matching_files[1])[1]
            metadata_crushed_2023_ITS_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files[0].split("/")[-1])[0]
            metadata_crushed_2023_ITS_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + ending_R1
            metadata_crushed_2023_ITS_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + ending_R2

        # else:
        #     # prefix2=prefix.replace("crushed-1-", "").replace("-", "_")
        #     prefix2=prefix.replace("washed-1-", "")#.replace("-", "_")
        #     matching_files2 = list_files_with_prefix(folder_path, prefix2)
    
        #     if len(matching_files2) == 2:
        #         metadata_crushed_2023_ITS_full.loc[index, "forward_fq"] = matching_files2[0]
        #         metadata_crushed_2023_ITS_full.loc[index, "reverse_fq"] = matching_files2[1]
        #         metadata_crushed_2023_ITS_full.loc[index, "current_seq_sample_ID"] = re.split(r"_S\d+", matching_files2[0].split("/")[-1])[0]
        #         metadata_crushed_2023_ITS_full.loc[index, "corrected_forward_fq"] = correct_seq_sample_ID + "_S" + re.split(r"_S\d+", matching_files2[0])[1]
        #         metadata_crushed_2023_ITS_full.loc[index, "corrected_reverse_fq"] = correct_seq_sample_ID + "_S" + re.split(r"_S\d+", matching_files2[1])[1]
        #     else:
        #         print(f"Files in {folder_path} containing {prefix2}:")
            
            # print(correct_seq_sample_ID)
            # print(prefix)
            # print()

        #         print(f" - {prefix}")
        #         print(f" - {matching_files}")

    except Exception as e:
        # Print the error and the problematic row
        print(f"Error occurred for row {index}: {row}")
        print(f"Error details: {e}")

    # Print the percentage of completion
    percent_complete = (n + 1) / total_rows * 100
    print(f"Processing... {percent_complete:.2f}% complete", end="\r")
    n=n+1

# After the loop, print 100% to indicate completion
print("\nProcessing complete!")
# crushed-1 SAMPLES, what to to with "correct_seq_sample_ID

Processing... 100.00% complete
Processing complete!


In [ ]:
columns_to_move=["current_seq_sample_ID","correct_seq_sample_ID", "suggested_correct_seq_sample_ID"]
# Reorder the DataFrame so that the specified columns are moved to the end
metadata_crushed_2023_ITS_full = metadata_crushed_2023_ITS_full[
    [col for col in metadata_crushed_2023_ITS_full.columns if col not in columns_to_move] + columns_to_move
]

# Extract only the file names from the 'forward_fq' column (removes the path)
metadata_crushed_2023_ITS_full['forward_fq'] = metadata_crushed_2023_ITS_full['forward_fq'].str.split("/").str[-1]

# Extract only the file names from the 'reverse_fq' column (removes the path)
metadata_crushed_2023_ITS_full['reverse_fq'] = metadata_crushed_2023_ITS_full['reverse_fq'].str.split("/").str[-1]

# Create a new column "SUGGESTED_EQUALS_CORRECT" that is True if the suggested correct seq_sample_ID equals the correct seq_sample_ID
metadata_crushed_2023_ITS_full["SUGGESTED_EQUALS_CORRECT"] = metadata_crushed_2023_ITS_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2023_ITS_full["correct_seq_sample_ID"]

# Create a new column "ALREADY_CORRECT" that is True if the suggested correct seq_sample_ID equals the current seq_sample_ID
metadata_crushed_2023_ITS_full["ALREADY_CORRECT"] = metadata_crushed_2023_ITS_full["suggested_correct_seq_sample_ID"] == metadata_crushed_2023_ITS_full["current_seq_sample_ID"]

# Sort the DataFrame by "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" columns
# The rows with "True" for "SUGGESTED_EQUALS_CORRECT" and "ALREADY_CORRECT" will come first
metadata_crushed_2023_ITS_full = metadata_crushed_2023_ITS_full.sort_values(["SUGGESTED_EQUALS_CORRECT", "ALREADY_CORRECT"])
metadata_crushed_2023_ITS_full


,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,spray_fungicide,fertiliser,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT
110,T-300523-c-3A-1,1.0,3.0,2023-05-30,timeline,1.0,3.0,1.0,n,Normal,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-300523-ITS-c-3A-1_S17_L001_R1_001.fastq.gz,T-300523-ITS-c-3A-1_S17_L001_R2_001.fastq.gz,T-300523-ITS-c-3A-1_S17_L001_R1_001.fastq.gz,T-300523-ITS-c-3A-1_S17_L001_R2_001.fastq.gz,T-300523-ITS-c-3A-1,NaN,T-300523-ITS-c-3A-1,False,True
141,T-300523-c-91A-1,32.0,91.0,2023-05-30,timeline,3.0,91.0,8.0,n,Zero,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-300523-ITS-c-91A-1_S36_L001_R1_001.fastq.gz,T-300523-ITS-c-91A-1_S36_L001_R2_001.fastq.gz,T-300523-ITS-c-91A-1_S36_L001_R1_001.fastq.gz,T-300523-ITS-c-91A-1_S36_L001_R2_001.fastq.gz,T-300523-ITS-c-91A-1,NaN,T-300523-ITS-c-91A-1,False,True
142,T-300523-c-98A-1,33.0,98.0,2023-05-30,timeline,3.0,98.0,9.0,n,Normal,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-300523-ITS-c-98A-1_S37_L001_R1_001.fastq.gz,T-300523-ITS-c-98A-1_S37_L001_R2_001.fastq.gz,T-300523-ITS-c-98A-1_S37_L001_R1_001.fastq.gz,T-300523-ITS-c-98A-1_S37_L001_R2_001.fastq.gz,T-300523-ITS-c-98A-1,NaN,T-300523-ITS-c-98A-1,False,True
143,T-300523-c-100A-1,34.0,100.0,2023-05-30,timeline,3.0,100.0,9.0,n,Normal,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-300523-ITS-c-100A-1_S38_L001_R1_001.fastq.gz,T-300523-ITS-c-100A-1_S38_L001_R2_001.fastq.gz,T-300523-ITS-c-100A-1_S38_L001_R1_001.fastq.gz,T-300523-ITS-c-100A-1_S38_L001_R2_001.fastq.gz,T-300523-ITS-c-100A-1,NaN,T-300523-ITS-c-100A-1,False,True
144,T-300523-c-104A-1,35.0,104.0,2023-05-30,timeline,3.0,104.0,9.0,n,Normal,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-300523-ITS-c-104A-1_S39_L001_R1_001.fastq.gz,T-300523-ITS-c-104A-1_S39_L001_R2_001.fastq.gz,T-300523-ITS-c-104A-1_S39_L001_R1_001.fastq.gz,T-300523-ITS-c-104A-1_S39_L001_R2_001.fastq.gz,T-300523-ITS-c-104A-1,NaN,T-300523-ITS-c-104A-1,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-ITS-c-redo-neg_S47_L001_R1_001.fastq.gz,T-2023-ITS-c-redo-neg_S47_L001_R2_001.fastq.gz,T-000023-ITS-c-redo-neg_S47_L001_R1_001.fastq.gz,T-000023-ITS-c-redo-neg_S47_L001_R2_001.fastq.gz,T-2023-ITS-c-redo-neg,T-000023-ITS-c-redo-neg,T-000023-ITS-c-redo-neg,True,False
92,Ultrapure-water,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-ITS-c-P1-neg_S93_L001_R1_001.fastq.gz,T-2023-ITS-c-P1-neg_S93_L001_R2_001.fastq.gz,T-000023-ITS-c-P1-neg_S93_L001_R1_001.fastq.gz,T-000023-ITS-c-P1-neg_S93_L001_R2_001.fastq.gz,T-2023-ITS-c-P1-neg,T-000023-ITS-c-P1-neg,T-000023-ITS-c-P1-neg,True,False
93,ZymoBIOMICS-Microbial-Community-DNA-Standard-D...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-ITS-c-P1-pos_S94_L001_R1_001.fastq.gz,T-2023-ITS-c-P1-pos_S94_L001_R2_001.fastq.gz,T-000023-ITS-c-P1-pos_S94_L001_R1_001.fastq.gz,T-000023-ITS-c-P1-pos_S94_L001_R2_001.fastq.gz,T-2023-ITS-c-P1-pos,T-000023-ITS-c-P1-pos,T-000023-ITS-c-P1-pos,True,False
477,Ultrapure-water,NaN,NaN,NaT,timeline,NaN,NaN,NaN,NaN,NaN,...,/home/jovyan/work/CCRP/MATRIX/RawData/Microbio...,T-2023-ITS-c-P5-neg_S95_L001_R1_001.fastq.gz,T-2023-ITS-c-P5-neg_S95_L001_R2_001.fastq.gz,T-000023-ITS-c-P5-neg_S95_L001_R1_001.fastq.gz,T-000023-ITS-c-P5-neg_S95_L001_R2_001.fastq.gz,T-2023-ITS-c-P5-neg,T-000023-ITS-c-P5-neg,T-000023-ITS-c-P5-neg,True,False


In [ ]:
filtered_metadata_crushed_2023_ITS_full=metadata_crushed_2023_ITS_full[
    metadata_crushed_2023_ITS_full.apply(
        lambda row: row['amplicon'] not in row['suggested_correct_seq_sample_ID'], axis=1)]
filtered_metadata_crushed_2023_ITS_full

,bio_sample_ID,vila_ID,sample_number,sample_date,sample_type,block,plot,subplot,spray_fungicide,fertiliser,...,full_path,forward_fq,reverse_fq,corrected_forward_fq,corrected_reverse_fq,current_seq_sample_ID,correct_seq_sample_ID,suggested_correct_seq_sample_ID,SUGGESTED_EQUALS_CORRECT,ALREADY_CORRECT


In [ ]:
metadata_crushed_seq_controls_full.to_csv("metadata_crushed_seq_controls.csv", index=False)
metadata_shotgun_2022_amplicon_full.to_csv("metadata_shotgun_2022_amplicon.csv", index=False)
metadata_shotgun_2022_full.to_csv("metadata_shotgun_2022.csv", index=False)
metadata_shotgun_2023_amplicon_full.to_csv("metadata_shotgun_2023_amplicon.csv", index=False)
metadata_shotgun_2023_full.to_csv("metadata_shotgun_2023.csv", index=False)
metadata_washed_2020_full.to_csv("metadata_washed_2020.csv", index=False)
metadata_washed_2022_full.to_csv("metadata_washed_2022.csv", index=False)
metadata_crushed_2021_full.to_csv("metadata_crushed_2021.csv", index=False)
metadata_crushed_2022_full.to_csv("metadata_crushed_2022.csv", index=False)
metadata_crushed_2023_16S_full.to_csv("metadata_crushed_2023_16S.csv", index=False)
metadata_crushed_2023_ITS_full.to_csv("metadata_crushed_2023_ITS.csv", index=False)